In [ ]:
#Eeg preprocessing qc

In [1]:
"""
EEG Preprocessing Quality-Control (QC) Diagnostic Suite
=========================================================

Purpose
-------
Given a RAW (unprocessed) EEG recording and its PREPROCESSED counterpart (same
subject, same recording -- e.g. output of a notch+bandpass+ICA+re-reference
pipeline), this script runs a battery of standard, literature-established
statistical tests and diagnostic plots to answer two questions:

  1) Did preprocessing actually work (is the cleaned signal statistically
     "better" than the raw one on every axis that matters)?
  2) If something still looks wrong, WHICH artifact/noise source is most
     likely responsible (line noise, eye blinks/EOG, muscle/EMG, heartbeat/ECG,
     electrode drift/flat channels, motion, bad channels, non-stationarity)?

Input
-----
Two file paths pointing to EEG recordings MNE can read (.fif, .edf, .bdf, .set,
.vhdr, or a .mat/.npy handled by the loader below), OR two already-loaded
mne.io.Raw objects. See `load_raw()`.

Output
------
- A structured, printed QC report (per-metric PASS/WARN/FAIL with the specific
  artifact it implicates).
- A `results` dict (JSON-serializable) with every computed metric.
- A multi-panel PNG figure per diagnostic category (PSD, topomap, ICA,
  stationarity, correlation, Hjorth, per-channel boxplots).

Dependencies
------------
mne, mne_icalabel (optional, for automatic IC classification), numpy, scipy,
matplotlib, seaborn, statsmodels (for the ADF stationarity test).

Usage
-----
    python eeg_preprocessing_qc.py --raw path/to/raw.fif --clean path/to/clean.fif \
        --out ./qc_report

Or import and call `run_qc(raw, clean, out_dir="qc_report")` from your own code.
"""

import os
import json
import warnings
import argparse
from dataclasses import dataclass, field

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import mne
import scipy.io as sio
from scipy import stats, signal
from scipy.stats import kurtosis, skew, shapiro, kstest, wilcoxon, ttest_rel, entropy as scipy_entropy

try:
    from statsmodels.tsa.stattools import adfuller
    HAS_STATSMODELS = True
except ImportError:
    HAS_STATSMODELS = False

try:
    import mne_icalabel
    HAS_ICALABEL = True
except ImportError:
    HAS_ICALABEL = False

try:
    import seaborn as sns
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------------------
# CONFIGURATION -- thresholds are literature-typical defaults; tune per dataset
# ----------------------------------------------------------------------------

LINE_FREQS = [50.0, 60.0]           # check both in case region is unknown
LINE_BAND_HALFWIDTH = 1.0           # Hz, window around the line frequency
LINE_NOISE_SNR_DB_THRESHOLD = 6.0   # dB peak-to-neighbourhood; above => line noise present

EEG_BANDS = {
    "delta": (0.5, 4.0),
    "theta": (4.0, 8.0),
    "alpha": (8.0, 13.0),
    "beta":  (13.0, 30.0),
    "gamma": (30.0, 45.0),
    "muscle_hf": (45.0, 100.0),     # EMG/muscle artifact typically dominates here
}

FLATLINE_STD_UV_THRESHOLD = 0.05    # microvolt std below this => dead/flat channel
CLIPPING_SATURATION_FRACTION = 0.01 # fraction of samples at min/max => clipping
AMPLITUDE_REJECT_UV = 150.0         # generic "gross artifact" peak-to-peak threshold
KURTOSIS_ARTIFACT_THRESHOLD = 5.0   # excess kurtosis above this suggests spiky artifacts (eye blinks/EMG)
CORR_LOW_THRESHOLD = 0.4            # inter-channel correlation below this (median) suggests bad channel/noise
ADF_PVALUE_THRESHOLD = 0.05         # ADF test: p < 0.05 => stationary (good)
SHAPIRO_ALPHA = 0.05                # normality test alpha (informational, EEG is rarely perfectly Gaussian)

# ----------------------------------------------------------------------------
# DATASET-SPECIFIC CONFIG (Shin et al. 2018 n-back, same layout/format as your
# preprocessing script). Update these two paths so they match your
# preprocessing script's BASE_DATASET_PATH / OUTPUT_PATH exactly -- everything
# else (subject folder naming, file names) is copied verbatim from your
# load_eeg_data() / process_subject() so the QC script reads the SAME raw
# files and the SAME preprocessed outputs your pipeline produces, with no
# manual path-typing needed per subject.
# ----------------------------------------------------------------------------
BASE_DATASET_PATH = r"E:\kntu\Thesis\codes\DATA"                              # NEW: same as preprocessing script's BASE_DATASET_PATH
OUTPUT_PATH       = r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"   # NEW: same as preprocessing script's OUTPUT_PATH
SFREQ_EEG         = 200.0                                                     # NEW: same as preprocessing script's SFREQ_EEG
EEG_BAND_ORDER     = ["delta", "theta", "alpha", "beta", "gamma"]             # NEW: must match preprocessing script's EEG_BAND_ORDER


def build_real_eeg_montage(subject_id, ch_names):
    """
    FIX (per user correction): the dataset provides REAL electrode 3D positions in
    BASE_DATASET_PATH\\{subject_id}-EEG\\mnt_nback.mat (fields .pos_3d shape (3,30), .clab shape
    (30,)) -- this is the SAME file your preprocessing script's get_eeg_mnt() reads for
    spatial_mapping(). The generic 'standard_1005' template used previously is a guessed template
    montage, NOT this dataset's real electrode layout, and should not be used for topomap/ICA
    plotting when the real positions are available.

    This builds a proper mne.channels.DigMontage from mnt_nback.mat, matched BY NAME to `ch_names`
    (same name-matching approach as get_eeg_mnt() in your preprocessing script, since channel order
    in cnt_nback.mat vs mnt_nback.mat is not guaranteed identical). Returns None (caller should
    skip set_montage) if mnt_nback.mat is missing or no channels match.
    """
    eeg_folder = os.path.join(BASE_DATASET_PATH, f"{subject_id}-EEG")
    mnt_path = os.path.join(eeg_folder, "mnt_nback.mat")
    if not os.path.exists(mnt_path):
        print(f"  WARNING: {mnt_path} not found -- cannot build a real montage for {subject_id}, "
              f"topomap/ICA plots will be skipped.")
        return None

    mat_mnt = sio.loadmat(mnt_path, struct_as_record=False, squeeze_me=True)
    real_mnt_keys = [k for k in mat_mnt.keys() if not k.startswith("__")]
    mnt_struct = mat_mnt[real_mnt_keys[0]]

    full_clab = [str(c).strip() for c in np.asarray(mnt_struct.clab).flatten()]
    full_pos_3d = np.asarray(mnt_struct.pos_3d, dtype=np.float64)  # (3, n_channels_in_mnt)
    name_to_idx = {name: i for i, name in enumerate(full_clab)}
    name_to_idx_upper = {name.upper(): i for i, name in enumerate(full_clab)}

    ch_pos = {}
    for ch in ch_names:
        idx = name_to_idx.get(ch, name_to_idx_upper.get(ch.upper(), None))
        if idx is not None:
            ch_pos[ch] = full_pos_3d[:, idx]

    if not ch_pos:
        print(f"  WARNING: no channel names in {ch_names} matched mnt_nback.mat's clab for "
              f"{subject_id} -- topomap/ICA plots will be skipped.")
        return None

    montage = mne.channels.make_dig_montage(ch_pos=ch_pos, coord_frame="head")
    return montage


def load_raw_from_dataset(subject_id):
    """
    Loads the RAW (unprocessed) continuous EEG for `subject_id` (e.g. "VP001")
    directly from BASE_DATASET_PATH\\{subject_id}-EEG\\cnt_nback.mat, using the
    SAME reading logic as load_eeg_data() in your preprocessing script (same
    struct fields, same unit auto-detection, same channel typing). Returns an
    mne.io.Raw object -- no filtering/ICA/re-referencing applied (this is the
    RAW arm of the comparison).
    """
    eeg_folder = os.path.join(BASE_DATASET_PATH, f"{subject_id}-EEG")
    cnt_path = os.path.join(eeg_folder, "cnt_nback.mat")
    if not os.path.exists(cnt_path):
        raise FileNotFoundError(f"Raw EEG file not found: {cnt_path}")

    mat_cnt = sio.loadmat(cnt_path, struct_as_record=False, squeeze_me=True)
    real_cnt_keys = [k for k in mat_cnt.keys() if not k.startswith("__")]
    cnt_struct = mat_cnt[real_cnt_keys[0]]

    data = cnt_struct.x.T
    sfreq = float(cnt_struct.fs)
    ch_names = [str(ch).strip() for ch in cnt_struct.clab]

    if np.max(np.abs(data)) > 1.0:
        data = data * 1e-6  # match preprocessing script's uV -> V auto-detection

    if sfreq != SFREQ_EEG:
        n_samples = int(data.shape[1] * SFREQ_EEG / sfreq)
        data = signal.resample(data, n_samples, axis=1)
        sfreq = SFREQ_EEG

    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types="eeg")
    raw = mne.io.RawArray(data, info, verbose=False)
    eog_channels = [ch for ch in ["VEOG", "HEOG"] if ch in ch_names]
    if eog_channels:
        raw.set_channel_types({ch: "eog" for ch in eog_channels})
    # FIX (per user correction): use REAL electrode positions from this subject's own
    # mnt_nback.mat (same source your preprocessing script's spatial_mapping() uses), not the
    # generic 'standard_1005' template montage used previously.
    montage = build_real_eeg_montage(subject_id, raw.ch_names)  # FIX
    if montage is not None:  # FIX
        raw.set_montage(montage, match_case=False, on_missing="ignore")  # FIX
    return raw


def load_clean_from_pipeline(subject_id):
    """
    Loads the PREPROCESSED continuous EEG for `subject_id` from your pipeline's
    OUTPUT_PATH\\{subject_id}\\ folder. Two supported sources, tried in order:

    1) PRIMARY / RECOMMENDED: a continuous, channel-space, post-ICA .fif file
       named "{subject_id}_eeg_clean_raw.fif". Your preprocessing script does
       NOT currently save this -- add ONE line to process_subject(), right
       after `raw_eeg_clean = preprocess_eeg(raw_eeg, events)`:

           raw_eeg_clean.save(os.path.join(sub_out_dir, f"{subject_id}_eeg_clean_raw.fif"),
                               overwrite=True)

       (sub_out_dir is created a few lines below that point in your script;
       if you add the save line before sub_out_dir exists, move the
       os.makedirs(sub_out_dir, exist_ok=True) line up, or just os.makedirs
       here too.) This gives an exact, non-lossy comparison against raw.

    2) FALLBACK (approximate): if the .fif above doesn't exist, this function
       reconstructs a pseudo-continuous signal from the saved windowed,
       5-band, spatially-mapped tensor "{subject_id}_eeg_windows.npy"
       (shape (N,16,16,5,600)) by averaging over the 16x16 spatial grid to
       get back to a coarse "virtual channel" per band, then concatenating
       windows along time. This is LOSSY (spatial detail is destroyed by the
       averaging, and windows are concatenated with their 66% overlap simply
       ignored) and is only good enough for spectral/statistical sanity
       checks -- NOT a substitute for source (1) above. A clear WARNING is
       printed when this fallback is used.
    """
    sub_dir = os.path.join(OUTPUT_PATH, subject_id)

    fif_path = os.path.join(sub_dir, f"{subject_id}_eeg_clean_raw.fif")
    if os.path.exists(fif_path):
        clean_raw = mne.io.read_raw_fif(fif_path, preload=True, verbose=False)
        # FIX (per user correction): raw_eeg_clean in your preprocessing pipeline was originally
        # built from load_eeg_data(), which sets the generic 'standard_1005' template montage (kept
        # there only for interpolate_bads()'s internal use, per your pipeline's own comments) -- NOT
        # this subject's real electrode positions. Re-apply the REAL montage from mnt_nback.mat here
        # so topomap/ICA plotting on the preprocessed data uses actual electrode geometry, exactly
        # matching what spatial_mapping()/get_eeg_mnt() use in your preprocessing script.
        montage = build_real_eeg_montage(subject_id, clean_raw.ch_names)  # FIX
        if montage is not None:  # FIX
            clean_raw.set_montage(montage, match_case=False, on_missing="ignore")  # FIX
        return clean_raw

    windows_path = os.path.join(sub_dir, f"{subject_id}_eeg_windows.npy")
    if not os.path.exists(windows_path):
        raise FileNotFoundError(
            f"Neither '{fif_path}' nor '{windows_path}' found for {subject_id}. "
            f"Check OUTPUT_PATH and that preprocessing has been run for this subject."
        )

    print(f"  WARNING: '{subject_id}_eeg_clean_raw.fif' not found -- falling back to an "
          f"APPROXIMATE reconstruction from windowed data ({windows_path}). "
          f"See load_clean_from_pipeline() docstring to enable the exact comparison instead.")

    windows = np.load(windows_path)  # (N, 16, 16, 5, 600)
    n_windows, gH, gW, n_bands, n_times = windows.shape
    # Average over the spatial grid -> one pseudo-channel per band, then treat each band as a
    # "channel" and concatenate windows along time (ignoring overlap) to get a long pseudo-signal.
    pseudo = windows.mean(axis=(1, 2))               # (N, n_bands, n_times)
    pseudo = np.transpose(pseudo, (1, 0, 2))          # (n_bands, N, n_times)
    pseudo = pseudo.reshape(n_bands, n_windows * n_times)  # (n_bands, N*n_times)

    ch_names = [f"pseudo_{b}" for b in EEG_BAND_ORDER[:n_bands]]
    info = mne.create_info(ch_names=ch_names, sfreq=SFREQ_EEG, ch_types="eeg")
    return mne.io.RawArray(pseudo.astype(np.float64), info, verbose=False)


# ----------------------------------------------------------------------------
# DATA STRUCTURES
# ----------------------------------------------------------------------------

@dataclass
class QCResult:
    metric: str
    raw_value: float = None
    clean_value: float = None
    status: str = "INFO"           # PASS / WARN / FAIL / INFO
    likely_artifact: str = ""
    detail: str = ""

    def to_dict(self):
        return dict(metric=self.metric, raw_value=self.raw_value,
                     clean_value=self.clean_value, status=self.status,
                     likely_artifact=self.likely_artifact, detail=self.detail)


# ----------------------------------------------------------------------------
# I/O
# ----------------------------------------------------------------------------

def load_raw(path_or_raw, sfreq_if_array=250.0, ch_names=None):
    """
    Load an EEG recording into an mne.io.Raw object.
    Accepts: a path (.fif/.edf/.bdf/.set/.vhdr), an mne Raw object already in
    memory, or a numpy array of shape (n_channels, n_times) (in which case a
    generic EEG montage is created).
    """
    if isinstance(path_or_raw, mne.io.BaseRaw):
        return path_or_raw

    if isinstance(path_or_raw, np.ndarray):
        n_ch = path_or_raw.shape[0]
        names = ch_names or [f"EEG{i+1:03d}" for i in range(n_ch)]
        info = mne.create_info(names, sfreq_if_array, ch_types="eeg")
        return mne.io.RawArray(path_or_raw, info, verbose=False)

    ext = os.path.splitext(path_or_raw)[1].lower()
    if ext == ".fif":
        return mne.io.read_raw_fif(path_or_raw, preload=True, verbose=False)
    elif ext in (".edf",):
        return mne.io.read_raw_edf(path_or_raw, preload=True, verbose=False)
    elif ext in (".bdf",):
        return mne.io.read_raw_bdf(path_or_raw, preload=True, verbose=False)
    elif ext == ".set":
        return mne.io.read_raw_eeglab(path_or_raw, preload=True, verbose=False)
    elif ext == ".vhdr":
        return mne.io.read_raw_brainvision(path_or_raw, preload=True, verbose=False)
    elif ext == ".npy":
        arr = np.load(path_or_raw)
        return load_raw(arr, sfreq_if_array=sfreq_if_array, ch_names=ch_names)
    else:
        raise ValueError(f"Unsupported file extension: {ext}")


def _get_data_uv(raw):
    """Return data in microvolts, shape (n_channels, n_times)."""
    data = raw.get_data(picks="eeg")
    # MNE stores EEG in volts; heuristically detect and convert to microvolts.
    if np.median(np.abs(data)) < 1e-3:
        data = data * 1e6
    return data


# ----------------------------------------------------------------------------
# 1) BASIC SIGNAL-QUALITY METRICS (amplitude / flatline / clipping / kurtosis)
# ----------------------------------------------------------------------------

def check_amplitude_and_flatline(raw, label, results, epoch_seconds=2.0):
    data = _get_data_uv(raw)
    ch_names = raw.ch_names[:data.shape[0]]

    stds = data.std(axis=1)
    ptp = np.ptp(data, axis=1)  # whole-recording per-channel ptp -- kept for the flat/clip checks below,
    # which legitimately want a whole-recording view (a channel is "dead" or "saturated" as a whole).
    kurt = kurtosis(data, axis=1, fisher=True)  # excess kurtosis; ~0 for Gaussian, high for spiky artifacts
    skewness = skew(data, axis=1)

    flat_channels = [ch_names[i] for i, s in enumerate(stds) if s < FLATLINE_STD_UV_THRESHOLD]
    high_kurt_channels = [ch_names[i] for i, k in enumerate(kurt) if k > KURTOSIS_ARTIFACT_THRESHOLD]

    # clipping / saturation: fraction of samples at the extreme min/max of each channel
    clip_fracs = []
    for i in range(data.shape[0]):
        ch = data[i]
        lo, hi = ch.min(), ch.max()
        frac = np.mean((ch <= lo + 1e-9) | (ch >= hi - 1e-9))
        clip_fracs.append(frac)
    clipped_channels = [ch_names[i] for i, f in enumerate(clip_fracs) if f > CLIPPING_SATURATION_FRACTION]

    # FIX (per chat history -- real bug, not a preprocessing issue): the previous version computed
    # np.ptp(data, axis=1), which is the peak-to-peak of each channel over the ENTIRE continuous
    # recording, then averaged across CHANNELS -- despite being labelled "fraction of gross-
    # amplitude EPOCHS". A single transient (one imperfectly-removed blink, one brief movement)
    # anywhere in a multi-minute recording makes that channel's whole-recording ptp exceed the
    # threshold, so this metric was almost guaranteed to read high even for well-cleaned data --
    # it never actually measured what fraction of short time windows are contaminated, which is
    # what "epoch rejection" means in the EEG literature (e.g. the fixed-length epoching used by
    # autoreject / FASTER-style pipelines). Genuine epoch-based rejection below: the continuous
    # signal is cut into non-overlapping `epoch_seconds`-long windows, ptp is computed per
    # (epoch, channel), and the fraction of epoch-channel cells exceeding the threshold is reported
    # -- this is the metric the label always claimed to be.
    sfreq = raw.info["sfreq"]
    n_per_epoch = int(epoch_seconds * sfreq)
    n_epochs = data.shape[1] // n_per_epoch
    if n_epochs >= 1:
        epoched = data[:, :n_epochs * n_per_epoch].reshape(data.shape[0], n_epochs, n_per_epoch)
        epoch_ptp = np.ptp(epoched, axis=2)  # (n_channels, n_epochs)
        gross_artifact_frac = np.mean(epoch_ptp > AMPLITUDE_REJECT_UV)  # FIX: real epoch-channel fraction
    else:
        gross_artifact_frac = float(np.mean(ptp > AMPLITUDE_REJECT_UV))  # fallback for very short signals

    results.append(QCResult(
        metric=f"{label}: flat/dead channels (std < {FLATLINE_STD_UV_THRESHOLD} uV)",
        raw_value=len(flat_channels), status="FAIL" if flat_channels else "PASS",
        likely_artifact="Electrode disconnection / dead channel" if flat_channels else "",
        detail=str(flat_channels)))

    results.append(QCResult(
        metric=f"{label}: clipped/saturated channels (>{CLIPPING_SATURATION_FRACTION*100:.1f}% samples at extremes)",
        raw_value=len(clipped_channels), status="FAIL" if clipped_channels else "PASS",
        likely_artifact="Amplifier saturation / electrode-skin contact failure" if clipped_channels else "",
        detail=str(clipped_channels)))

    results.append(QCResult(
        metric=f"{label}: high-kurtosis channels (excess kurtosis > {KURTOSIS_ARTIFACT_THRESHOLD})",
        raw_value=len(high_kurt_channels), status="WARN" if high_kurt_channels else "PASS",
        likely_artifact="Spiky transient artifacts (eye blinks, electrode pops, EMG bursts)" if high_kurt_channels else "",
        detail=str(high_kurt_channels)))

    results.append(QCResult(
        metric=f"{label}: fraction of gross-amplitude epochs (>{AMPLITUDE_REJECT_UV} uV p2p, "
               f"{epoch_seconds:.0f}s epochs)",  # FIX: label now matches what's actually computed
        raw_value=float(gross_artifact_frac),
        status="FAIL" if gross_artifact_frac > 0.10 else ("WARN" if gross_artifact_frac > 0.02 else "PASS"),
        likely_artifact="Movement artifact / muscle burst / electrode pop" if gross_artifact_frac > 0.02 else "",
        detail=f"mean std={stds.mean():.2f}uV, mean whole-recording ptp={ptp.mean():.2f}uV, "
               f"mean skew={skewness.mean():.2f}"))

    return dict(stds=stds, ptp=ptp, kurt=kurt, skew=skewness, clip_fracs=np.array(clip_fracs))


# ----------------------------------------------------------------------------
# 2) LINE NOISE (50/60 Hz) DETECTION -- Welch PSD peak test
# ----------------------------------------------------------------------------

def check_line_noise(raw, label, results):
    data = _get_data_uv(raw)
    sfreq = raw.info["sfreq"]
    freqs, psd = signal.welch(data, fs=sfreq, nperseg=min(int(4 * sfreq), data.shape[1]), axis=1)
    psd_db = 10 * np.log10(psd.mean(axis=0) + 1e-20)

    # FIX (per chat history -- real bug, not a preprocessing issue): if `raw` has already been
    # lowpass-filtered (MNE records this in raw.info['lowpass'] whenever .filter(h_freq=...) was
    # called), checking for line noise at a frequency ABOVE that lowpass cutoff means comparing two
    # regions that are BOTH deep in the filter's stopband -- i.e. both at the numerical attenuation
    # floor of the FIR filter, not at any real signal power. Any tiny difference in the filter's
    # roll-off shape between the "peak" window (49-51 Hz) and "neighbour" windows (45-48/52-55 Hz)
    # then reads as a spurious multi-dB "SNR", uniformly across every subject and, tellingly, at
    # BOTH 50 Hz AND 60 Hz roughly equally (both sit in the same already-attenuated region above a
    # 40 Hz cutoff) -- which is exactly the pattern observed in your VP001-VP011 results and is the
    # signature of a stopband-floor artifact, not genuine residual mains interference. This check
    # is only meaningful for signals whose passband still covers the probed frequency.
    lowpass = raw.info.get("lowpass", None)  # NEW

    line_hits = {}
    skipped_stopband = []  # NEW
    for lf in LINE_FREQS:
        if lf >= sfreq / 2:
            continue
        if lowpass is not None and (lf - LINE_BAND_HALFWIDTH) > lowpass:  # NEW
            skipped_stopband.append(lf)  # NEW: entire probe band lies above the filter's passband
            continue  # NEW
        band_mask = (freqs >= lf - LINE_BAND_HALFWIDTH) & (freqs <= lf + LINE_BAND_HALFWIDTH)
        neigh_mask = ((freqs >= lf - 5) & (freqs <= lf - 2)) | ((freqs >= lf + 2) & (freqs <= lf + 5))
        if not band_mask.any() or not neigh_mask.any():
            continue
        peak_db = psd_db[band_mask].max()
        neigh_db = psd_db[neigh_mask].mean()
        snr_db = peak_db - neigh_db
        line_hits[lf] = snr_db

    if not line_hits and skipped_stopband:  # NEW: every line frequency was above the filter's passband
        results.append(QCResult(  # NEW
            metric=f"{label}: line noise peak SNR (50/60 Hz)",  # NEW
            status="INFO",  # NEW
            detail=f"Skipped -- {skipped_stopband} Hz all lie above this signal's lowpass filter "  # NEW
                   f"cutoff ({lowpass} Hz), so any 'line noise' reading there would just be "  # NEW
                   f"filter stopband floor, not real interference."))  # NEW
        return freqs, psd_db  # NEW

    worst_freq, worst_snr = (max(line_hits.items(), key=lambda kv: kv[1])
                              if line_hits else (None, -np.inf))
    status = "FAIL" if worst_snr > LINE_NOISE_SNR_DB_THRESHOLD else "PASS"

    results.append(QCResult(
        metric=f"{label}: line noise peak SNR (50/60 Hz)",
        raw_value=float(worst_snr) if worst_snr != -np.inf else None,
        status=status,
        likely_artifact=f"Power-line interference ({worst_freq} Hz)" if status == "FAIL" else "",
        detail=str({k: round(v, 2) for k, v in line_hits.items()}) +
               (f" (skipped {skipped_stopband} Hz -- above lowpass cutoff {lowpass} Hz)" if skipped_stopband else "")))  # NEW

    return freqs, psd_db


# ----------------------------------------------------------------------------
# 3) BAND-POWER / SPECTRAL PROFILE + PAIRED STATISTICAL TESTS (raw vs clean)
# ----------------------------------------------------------------------------

def compute_band_power(raw):
    data = _get_data_uv(raw)
    sfreq = raw.info["sfreq"]
    freqs, psd = signal.welch(data, fs=sfreq, nperseg=min(int(4 * sfreq), data.shape[1]), axis=1)
    band_power = {}
    for band, (lo, hi) in EEG_BANDS.items():
        mask = (freqs >= lo) & (freqs <= hi)
        band_power[band] = psd[:, mask].mean(axis=1)  # per-channel mean power in band
    return band_power, freqs, psd


def compare_band_power(raw, clean, results):
    bp_raw, freqs_r, psd_r = compute_band_power(raw)
    bp_clean, freqs_c, psd_c = compute_band_power(clean)

    for band in EEG_BANDS:
        a = bp_raw[band]
        b = bp_clean[band]
        n = min(len(a), len(b))
        a, b = a[:n], b[:n]

        # Paired statistical test: Wilcoxon signed-rank (non-parametric, robust to
        # non-Gaussian EEG power distributions) is the primary test; paired t-test
        # reported alongside as the classical parametric counterpart.
        try:
            wilcoxon_stat, wilcoxon_p = wilcoxon(a, b)
        except ValueError:
            wilcoxon_stat, wilcoxon_p = np.nan, np.nan
        ttest_stat, ttest_p = ttest_rel(a, b)

        pct_change = 100 * (b.mean() - a.mean()) / (a.mean() + 1e-20)

        artifact_hint = ""
        if band == "muscle_hf" and pct_change < -20:
            artifact_hint = "EMG/muscle power reduced by cleaning (expected & good)"
        elif band == "muscle_hf" and pct_change > 20:
            artifact_hint = "Residual EMG/muscle artifact NOT removed by preprocessing"
        elif band in ("delta",) and pct_change > 50:
            artifact_hint = "Possible residual slow drift / sweat artifact in delta band"

        results.append(QCResult(
            metric=f"Band power [{band}] raw vs clean (Wilcoxon signed-rank)",
            raw_value=float(a.mean()), clean_value=float(b.mean()),
            status="INFO",
            likely_artifact=artifact_hint,
            detail=f"pct_change={pct_change:+.1f}%, wilcoxon_p={wilcoxon_p:.4g}, paired_t_p={ttest_p:.4g}"))

    return bp_raw, bp_clean, freqs_r, psd_r, freqs_c, psd_c


# ----------------------------------------------------------------------------
# 4) STATIONARITY -- Augmented Dickey-Fuller test (classic econometrics/signal
#    processing test; non-stationary EEG segments often indicate drift/motion)
# ----------------------------------------------------------------------------

def check_stationarity(raw, label, results, max_channels=8):
    if not HAS_STATSMODELS:
        results.append(QCResult(metric=f"{label}: stationarity (ADF test)", status="INFO",
                                 detail="statsmodels not installed -- skipped"))
        return None

    data = _get_data_uv(raw)
    n_ch = min(max_channels, data.shape[0])
    pvals = []
    for i in range(n_ch):
        # subsample for speed on long recordings
        seg = data[i, ::max(1, data.shape[1] // 20000)]
        try:
            adf_stat, pvalue, *_ = adfuller(seg, autolag="AIC")
        except Exception:
            pvalue = np.nan
        pvals.append(pvalue)
    pvals = np.array(pvals)
    frac_nonstationary = np.mean(pvals >= ADF_PVALUE_THRESHOLD)

    results.append(QCResult(
        metric=f"{label}: fraction of non-stationary channels (ADF p >= {ADF_PVALUE_THRESHOLD})",
        raw_value=float(frac_nonstationary),
        status="WARN" if frac_nonstationary > 0.3 else "PASS",
        likely_artifact="Slow drift / baseline wander / motion artifact" if frac_nonstationary > 0.3 else "",
        detail=f"mean ADF p-value={np.nanmean(pvals):.4g}"))
    return pvals


# ----------------------------------------------------------------------------
# 5) NORMALITY -- Shapiro-Wilk / Kolmogorov-Smirnov (informational: EEG amplitude
#    distributions deviating strongly from Gaussian often indicate artifact
#    contamination, since clean cortical EEG is approximately Gaussian at rest)
# ----------------------------------------------------------------------------

def check_normality(raw, label, results, max_channels=8, max_samples=5000):
    data = _get_data_uv(raw)
    n_ch = min(max_channels, data.shape[0])
    sw_pvals, ks_pvals = [], []
    for i in range(n_ch):
        seg = data[i]
        if len(seg) > max_samples:
            idx = np.random.choice(len(seg), max_samples, replace=False)
            seg = seg[idx]
        try:
            _, p_sw = shapiro(seg)
        except Exception:
            p_sw = np.nan
        _, p_ks = kstest((seg - seg.mean()) / (seg.std() + 1e-12), "norm")
        sw_pvals.append(p_sw)
        ks_pvals.append(p_ks)

    frac_nonnormal = np.mean(np.array(sw_pvals) < SHAPIRO_ALPHA)
    results.append(QCResult(
        metric=f"{label}: fraction of non-Gaussian channels (Shapiro-Wilk p < {SHAPIRO_ALPHA})",
        raw_value=float(frac_nonnormal), status="INFO",
        likely_artifact="Heavy-tailed/non-Gaussian amplitude distribution -- consistent with residual "
                         "blink/muscle spikes if fraction is high" if frac_nonnormal > 0.5 else "",
        detail=f"mean Shapiro p={np.nanmean(sw_pvals):.4g}, mean KS p={np.nanmean(ks_pvals):.4g}"))
    return sw_pvals, ks_pvals


# ----------------------------------------------------------------------------
# 6) INTER-CHANNEL CORRELATION -- low correlation flags noisy/bad channels;
#    abnormally HIGH uniform correlation across all channels flags a common
#    artifact source (e.g. unremoved eye-blink component riding on all channels)
# ----------------------------------------------------------------------------

def check_channel_correlation(raw, label, results):
    data = _get_data_uv(raw)
    n_ch = data.shape[0]
    corr = np.corrcoef(data)
    np.fill_diagonal(corr, np.nan)
    mean_corr_per_ch = np.nanmean(corr, axis=1)
    global_mean_corr = np.nanmean(corr)

    # FIX (per chat history -- real bug, not a preprocessing issue): common-average referencing
    # (CAR) forces sum_i x_i(t) = 0 at every timepoint, which mathematically drives the EXPECTED
    # mean pairwise correlation toward -1/(N-1) (e.g. -1/27 ~= -0.037 for 28 channels) even for a
    # perfectly healthy, well-preprocessed recording -- this is a property of the referencing
    # scheme, not evidence of noise. The original CORR_LOW_THRESHOLD=0.4 was calibrated for
    # un-referenced (or non-CAR) data and, when applied to CAR'd data, flagged nearly every channel
    # as "low correlation" in every subject -- exactly the uniform, all-channels-flagged pattern
    # seen across VP001-VP011. FIX: detect whether CAR was applied (via raw.info['custom_ref_applied']
    # if the .fif carries it, with a name-based EEG count fallback) and shift the low-correlation
    # threshold down by the expected CAR baseline instead of using a fixed absolute cutoff.
    is_car = bool(raw.info.get("custom_ref_applied", False))  # FIX: simplified, confirmed mne.Info.get() works directly
    expected_baseline = -1.0 / max(n_ch - 1, 1) if (is_car or n_ch > 1) else 0.0  # NEW
    # FIX (bug in the first version of this fix): "CORR_LOW_THRESHOLD - 0.4" always evaluated to
    # exactly 0 (since CORR_LOW_THRESHOLD's own default value IS 0.4), which put the effective
    # threshold EXACTLY AT the CAR baseline with zero tolerance -- flagging ~half of all channels
    # purely from random noise scattering around that baseline (confirmed by test: 12/28 channels
    # flagged on a fully synthetic, artifact-free signal). Use an explicit, independent margin
    # instead: a channel is flagged only if its correlation falls meaningfully BELOW the
    # theoretical CAR baseline, not merely at or slightly below it.
    CORR_MARGIN_BELOW_CAR_BASELINE = 0.15  # NEW: independent of CORR_LOW_THRESHOLD, tune if needed
    effective_threshold = (expected_baseline - CORR_MARGIN_BELOW_CAR_BASELINE) if is_car else CORR_LOW_THRESHOLD  # FIX

    low_corr_channels = [raw.ch_names[i] for i, c in enumerate(mean_corr_per_ch) if c < effective_threshold]  # CHANGED

    status = "WARN" if low_corr_channels else "PASS"
    artifact = ""
    if low_corr_channels:
        artifact = "Bad/noisy channel(s) poorly correlated with neighbours (possible electrode fault)"
    if global_mean_corr > 0.85:
        status = "WARN"
        artifact = (artifact + "; " if artifact else "") + \
            "Abnormally high global correlation -- possible unremoved common artifact " \
            "(eye blink / reference / movement affecting all channels similarly)"

    results.append(QCResult(
        metric=f"{label}: inter-channel correlation",
        raw_value=float(global_mean_corr), status=status,
        likely_artifact=artifact,
        detail=f"low-corr channels: {low_corr_channels} | "
               f"CAR-expected baseline: {expected_baseline:.3f}, effective threshold used: {effective_threshold:.3f}"))  # NEW
    return corr


# ----------------------------------------------------------------------------
# 7) HJORTH PARAMETERS (Activity/Mobility/Complexity) -- classic EEG QC metrics
# ----------------------------------------------------------------------------

def hjorth_parameters(data):
    """data: (n_channels, n_times). Returns activity, mobility, complexity per channel."""
    d1 = np.diff(data, axis=1)
    d2 = np.diff(d1, axis=1)
    var0 = np.var(data, axis=1)
    var1 = np.var(d1, axis=1)
    var2 = np.var(d2, axis=1)
    activity = var0
    mobility = np.sqrt(var1 / (var0 + 1e-20))
    complexity = np.sqrt(var2 / (var1 + 1e-20)) / (mobility + 1e-20)
    return activity, mobility, complexity


def check_hjorth(raw, label, results):
    data = _get_data_uv(raw)
    activity, mobility, complexity = hjorth_parameters(data)
    # Complexity >> 1.5-2 typically indicates irregular, artifact-like signal;
    # normal resting EEG complexity is roughly in the 1.0-1.5 range (dataset dependent).
    high_complexity_frac = np.mean(complexity > 2.0)
    results.append(QCResult(
        metric=f"{label}: Hjorth complexity (fraction of channels > 2.0)",
        raw_value=float(high_complexity_frac), status="INFO",
        likely_artifact="High signal irregularity -- consistent with EMG/muscle contamination" if high_complexity_frac > 0.3 else "",
        detail=f"mean activity={activity.mean():.2f}, mean mobility={mobility.mean():.3f}, "
               f"mean complexity={complexity.mean():.3f}"))
    return activity, mobility, complexity


# ----------------------------------------------------------------------------
# 8) SIGNAL-TO-NOISE RATIO IMPROVEMENT (raw vs clean) -- broadband RMS ratio in
#    the physiological band (1-40 Hz) vs the "noise-dominated" high band (>45 Hz)
# ----------------------------------------------------------------------------

def check_snr_improvement(raw, clean, results):
    def band_rms(raw_obj, lo, hi):
        d = raw_obj.copy().filter(lo, hi, method="fir", verbose=False).get_data(picks="eeg")
        return np.sqrt(np.mean(d ** 2))

    try:
        signal_rms_raw = band_rms(raw, 1.0, 40.0)
        noise_rms_raw = band_rms(raw, 45.0, min(99.0, raw.info["sfreq"] / 2 - 1))
        snr_raw = 20 * np.log10(signal_rms_raw / (noise_rms_raw + 1e-20))

        signal_rms_clean = band_rms(clean, 1.0, 40.0)
        noise_rms_clean = band_rms(clean, 45.0, min(99.0, clean.info["sfreq"] / 2 - 1))
        snr_clean = 20 * np.log10(signal_rms_clean / (noise_rms_clean + 1e-20))

        delta_snr = snr_clean - snr_raw
        results.append(QCResult(
            metric="Broadband SNR (1-40Hz signal / 45-100Hz noise band), dB",
            raw_value=float(snr_raw), clean_value=float(snr_clean),
            status="PASS" if delta_snr > 0 else "WARN",
            likely_artifact="Preprocessing did NOT improve SNR -- check filter settings / bad channels" if delta_snr <= 0 else "",
            detail=f"delta SNR = {delta_snr:+.2f} dB"))
    except Exception as e:
        results.append(QCResult(metric="Broadband SNR", status="INFO", detail=f"failed: {e}"))


# ----------------------------------------------------------------------------
# 9) ICA-BASED ARTIFACT SOURCE IDENTIFICATION (optional, needs mne_icalabel)
# ----------------------------------------------------------------------------

def check_ica_artifacts(clean, results, out_dir):
    if not HAS_ICALABEL:
        results.append(QCResult(metric="ICA component classification", status="INFO",
                                 detail="mne_icalabel not installed -- skipped"))
        return None

    try:
        clean_for_ica = clean.copy().filter(1.0, 40.0, verbose=False)
        clean_for_ica.set_eeg_reference("average", verbose=False)
        ica = mne.preprocessing.ICA(n_components=0.99, method="picard",
                                     fit_params=dict(extended=True), random_state=42, verbose=False)
        ica.fit(clean_for_ica, verbose=False)
        ic_labels = mne_icalabel.label_components(clean_for_ica, ica, method="iclabel")

        labels = ic_labels["labels"]
        probs = ic_labels["y_pred_proba"]
        counts = {}
        for lab in labels:
            counts[lab] = counts.get(lab, 0) + 1

        artifact_labels = {"eye blink", "muscle artifact", "heart beat", "line noise", "channel noise"}
        n_artifact = sum(v for k, v in counts.items() if k in artifact_labels)

        results.append(QCResult(
            metric="Residual ICA-detectable artifact components in PREPROCESSED data",
            raw_value=int(n_artifact), status="WARN" if n_artifact > 0 else "PASS",
            likely_artifact=", ".join(k for k in counts if k in artifact_labels and counts[k] > 0),
            detail=str(counts)))

        fig = ica.plot_components(show=False)
        figs = fig if isinstance(fig, list) else [fig]
        for i, f in enumerate(figs):
            f.savefig(os.path.join(out_dir, f"ica_components_{i}.png"), dpi=130)
            plt.close(f)

        return ic_labels
    except Exception as e:
        results.append(QCResult(metric="ICA component classification", status="INFO", detail=f"failed: {e}"))
        return None


# ----------------------------------------------------------------------------
# PLOTS
# ----------------------------------------------------------------------------

def plot_psd_comparison(freqs_r, psd_r, freqs_c, psd_c, out_dir):
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(freqs_r, psd_r, label="Raw", color="tab:red", alpha=0.8)
    ax.plot(freqs_c, psd_c, label="Preprocessed", color="tab:blue", alpha=0.8)
    for lf in LINE_FREQS:
        ax.axvline(lf, color="gray", linestyle="--", alpha=0.5)
    for band, (lo, hi) in EEG_BANDS.items():
        ax.axvspan(lo, hi, alpha=0.03)
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Power (dB, mean across channels)")
    ax.set_title("PSD: Raw vs Preprocessed (dashed = line-noise freqs, shaded = EEG bands)")
    ax.set_xlim(0, min(100, freqs_r.max()))
    ax.legend()
    plt.tight_layout()
    fig.savefig(os.path.join(out_dir, "psd_comparison.png"), dpi=150)
    plt.close(fig)


def plot_channel_boxplots(stats_raw, stats_clean, ch_names, out_dir):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    metrics = [("stds", "Std (uV)"), ("ptp", "Peak-to-peak (uV)"), ("kurt", "Excess kurtosis")]
    for ax, (key, title) in zip(axes, metrics):
        data_to_plot = [stats_raw[key], stats_clean[key]]
        ax.boxplot(data_to_plot, labels=["Raw", "Preprocessed"])
        ax.set_title(title)
    plt.tight_layout()
    fig.savefig(os.path.join(out_dir, "channel_quality_boxplots.png"), dpi=150)
    plt.close(fig)


def plot_correlation_heatmaps(corr_raw, corr_clean, out_dir):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, corr, title in zip(axes, [corr_raw, corr_clean], ["Raw", "Preprocessed"]):
        im = ax.imshow(np.nan_to_num(corr), vmin=-1, vmax=1, cmap="coolwarm")
        ax.set_title(f"Inter-channel correlation: {title}")
        plt.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout()
    fig.savefig(os.path.join(out_dir, "channel_correlation_heatmaps.png"), dpi=150)
    plt.close(fig)


def plot_topomap_band_power(clean, bp_clean, out_dir):
    try:
        eeg_picks = mne.pick_types(clean.info, eeg=True, eog=False)
        eeg_info = mne.pick_info(clean.info, eeg_picks)  # FIX: vals below only has one entry per EEG-type
        # channel (see compute_band_power's picks="eeg"), so the info object passed to plot_topomap
        # must be restricted the same way, or MNE raises a length-mismatch error whenever EOG/other
        # non-EEG channels are present in `clean` (e.g. VEOG/HEOG before they're dropped).
        n_bands = len(EEG_BANDS)
        fig, axes = plt.subplots(1, n_bands, figsize=(3 * n_bands, 3.5))
        for ax, (band, vals) in zip(axes, bp_clean.items()):
            mne.viz.plot_topomap(vals[:len(eeg_info["ch_names"])], eeg_info, axes=ax, show=False)  # FIX: eeg_info instead of clean.info
            ax.set_title(band)
        plt.tight_layout()
        fig.savefig(os.path.join(out_dir, "topomap_band_power.png"), dpi=150)
        plt.close(fig)
    except Exception as e:
        print(f"  [plot] topomap skipped (no valid montage / channel-position mismatch): {e}")


def plot_time_series_snippet(raw, clean, out_dir, duration=10.0, picks_n=5):
    sfreq = raw.info["sfreq"]
    n = int(duration * sfreq)
    data_r = _get_data_uv(raw)[:picks_n, :n]
    data_c = _get_data_uv(clean)[:picks_n, :min(n, _get_data_uv(clean).shape[1])]
    t_r = np.arange(data_r.shape[1]) / sfreq
    t_c = np.arange(data_c.shape[1]) / sfreq

    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
    offset = np.median(np.abs(data_r)) * 8 + 1e-6
    for i in range(data_r.shape[0]):
        axes[0].plot(t_r, data_r[i] + i * offset, linewidth=0.6, color="tab:red")
    axes[0].set_title(f"Raw (first {duration:.0f}s, {picks_n} channels)")
    for i in range(data_c.shape[0]):
        axes[1].plot(t_c, data_c[i] + i * offset, linewidth=0.6, color="tab:blue")
    axes[1].set_title(f"Preprocessed (first {duration:.0f}s, {picks_n} channels)")
    axes[1].set_xlabel("Time (s)")
    plt.tight_layout()
    fig.savefig(os.path.join(out_dir, "time_series_snippet.png"), dpi=150)
    plt.close(fig)


# ----------------------------------------------------------------------------
# REPORTING
# ----------------------------------------------------------------------------

def print_report(results):
    order = {"FAIL": 0, "WARN": 1, "INFO": 2, "PASS": 3}
    results_sorted = sorted(results, key=lambda r: order.get(r.status, 4))
    print("\n" + "=" * 100)
    print("EEG PREPROCESSING QC REPORT")
    print("=" * 100)
    for r in results_sorted:
        tag = f"[{r.status}]"
        print(f"{tag:<7} {r.metric}")
        if r.raw_value is not None or r.clean_value is not None:
            print(f"        raw={r.raw_value}  clean={r.clean_value}")
        if r.likely_artifact:
            print(f"        -> Likely artifact/cause: {r.likely_artifact}")
        if r.detail:
            print(f"        detail: {r.detail}")
        print("-" * 100)

    n_fail = sum(1 for r in results if r.status == "FAIL")
    n_warn = sum(1 for r in results if r.status == "WARN")
    print(f"\nSUMMARY: {n_fail} FAIL, {n_warn} WARN out of {len(results)} checks.")
    if n_fail == 0 and n_warn == 0:
        print("Preprocessing looks clean on all implemented checks.")
    elif n_fail > 0:
        print("Preprocessing has UNRESOLVED artifacts (see FAIL items above) -- do not proceed to modeling "
              "without addressing these.")
    else:
        print("Preprocessing is broadly acceptable but has minor residual issues (see WARN items).")


# ----------------------------------------------------------------------------
# MAIN ENTRY POINT
# ----------------------------------------------------------------------------

def run_qc(raw, clean, out_dir="qc_report"):
    os.makedirs(out_dir, exist_ok=True)
    raw = load_raw(raw)
    clean = load_raw(clean)

    results = []

    print("Running amplitude/flatline/clipping/kurtosis checks...")
    stats_raw = check_amplitude_and_flatline(raw, "RAW", results)
    stats_clean = check_amplitude_and_flatline(clean, "PREPROCESSED", results)

    print("Running line-noise (50/60 Hz) detection...")
    freqs_r, psd_r = check_line_noise(raw, "RAW", results)
    freqs_c, psd_c = check_line_noise(clean, "PREPROCESSED", results)

    print("Computing band power and paired statistical tests (Wilcoxon / paired t-test)...")
    bp_raw, bp_clean, freqs_r2, psd_r2, freqs_c2, psd_c2 = compare_band_power(raw, clean, results)

    print("Running stationarity test (Augmented Dickey-Fuller)...")
    check_stationarity(raw, "RAW", results)
    check_stationarity(clean, "PREPROCESSED", results)

    print("Running normality tests (Shapiro-Wilk / Kolmogorov-Smirnov)...")
    check_normality(raw, "RAW", results)
    check_normality(clean, "PREPROCESSED", results)

    print("Computing inter-channel correlation...")
    corr_raw = check_channel_correlation(raw, "RAW", results)
    corr_clean = check_channel_correlation(clean, "PREPROCESSED", results)

    print("Computing Hjorth parameters...")
    check_hjorth(raw, "RAW", results)
    check_hjorth(clean, "PREPROCESSED", results)

    print("Computing broadband SNR improvement...")
    check_snr_improvement(raw, clean, results)

    print("Running ICA-based residual-artifact classification on preprocessed data...")
    check_ica_artifacts(clean, results, out_dir)

    print("Generating plots...")
    plot_psd_comparison(freqs_r2, 10 * np.log10(psd_r2.mean(axis=0) + 1e-20),
                         freqs_c2, 10 * np.log10(psd_c2.mean(axis=0) + 1e-20), out_dir)
    plot_channel_boxplots(stats_raw, stats_clean, clean.ch_names, out_dir)
    plot_correlation_heatmaps(corr_raw, corr_clean, out_dir)
    plot_topomap_band_power(clean, bp_clean, out_dir)
    plot_time_series_snippet(raw, clean, out_dir)

    print_report(results)

    with open(os.path.join(out_dir, "qc_results.json"), "w") as f:
        json.dump([r.to_dict() for r in results], f, indent=2, default=str)

    print(f"\nAll plots and qc_results.json saved to: {os.path.abspath(out_dir)}")
    return results


def run_qc_for_subject(subject_id, out_root="qc_report"):
    """
    Convenience wrapper: loads RAW directly from BASE_DATASET_PATH and
    PREPROCESSED directly from OUTPUT_PATH for `subject_id` (e.g. "VP001"),
    using load_raw_from_dataset() / load_clean_from_pipeline() above, then
    runs the full QC suite and saves everything to out_root/{subject_id}/.
    """
    out_dir = os.path.join(out_root, subject_id)
    print(f"\n{'='*100}\nQC: {subject_id}\n{'='*100}")
    raw = load_raw_from_dataset(subject_id)
    clean = load_clean_from_pipeline(subject_id)
    return run_qc(raw, clean, out_dir=out_dir)


def run_qc_batch(subjects=None, out="qc_report", raw_path=None, clean_path=None):
    """
    Callable directly from a Jupyter cell (no argparse involved) -- this is what both the CLI
    branch below AND notebook users should call. Examples:

        # single manual raw/clean file pair:
        run_qc_batch(raw_path="C:/.../raw.fif", clean_path="C:/.../clean.fif", out="qc_report")

        # specific subjects from the dataset (uses BASE_DATASET_PATH / OUTPUT_PATH above):
        run_qc_batch(subjects=["VP001", "VP002"], out="qc_report")

        # every subject auto-discovered under BASE_DATASET_PATH:
        run_qc_batch(out="qc_report")
    """
    if raw_path and clean_path:
        return {"_manual_pair": [r.to_dict() for r in run_qc(raw_path, clean_path, out_dir=out)]}

    if subjects is None:
        subjects = sorted({
            name.split("-EEG")[0] for name in os.listdir(BASE_DATASET_PATH)
            if name.endswith("-EEG") and os.path.isdir(os.path.join(BASE_DATASET_PATH, name))
        })
        print(f"No subjects given -- auto-discovered {len(subjects)} subjects under "
              f"{BASE_DATASET_PATH}: {subjects}")

    all_results = {}
    for sid in subjects:
        try:
            all_results[sid] = [r.to_dict() for r in run_qc_for_subject(sid, out_root=out)]
        except Exception as e:
            print(f"  ERROR processing {sid}: {e}")

    os.makedirs(out, exist_ok=True)
    with open(os.path.join(out, "qc_all_subjects.json"), "w") as f:
        json.dump(all_results, f, indent=2, default=str)
    print(f"\nAll-subjects QC summary saved to: {os.path.join(out, 'qc_all_subjects.json')}")
    return all_results


if __name__ == "__main__":
    parser = argparse.ArgumentParser(
        description="EEG preprocessing QC diagnostic suite -- reads raw/preprocessed files "
                     "directly using the same paths/format as the preprocessing pipeline "
                     "(BASE_DATASET_PATH / OUTPUT_PATH, configured near the top of this file)."
    )
    parser.add_argument("--subjects", nargs="+", default=None,
                         help="Subject IDs to QC, e.g. --subjects VP001 VP002 VP003. "
                              "If omitted, QCs every VPxxx folder found under BASE_DATASET_PATH.")
    parser.add_argument("--out", default="qc_report", help="Output root directory (one subfolder per subject)")
    parser.add_argument("--raw", default=None, help="Manual path to a RAW EEG file (overrides --subjects)")
    parser.add_argument("--clean", default=None, help="Manual path to a PREPROCESSED EEG file (overrides --subjects)")
    # FIX (Jupyter/Windows compatibility): plain parser.parse_args() reads the full sys.argv,
    # which under `%run script.py` or `!python script.py` inside a Jupyter kernel also contains
    # the kernel's own arguments (e.g. "-f C:\...\kernel-xxxx.json"), causing argparse to raise
    # "unrecognized arguments" and SystemExit -- the script would crash before doing anything.
    # parse_known_args() ignores any argument it doesn't recognize instead of erroring out, so the
    # script runs cleanly both as a normal Windows command-line script (`python eeg_...py --out ...`)
    # and when executed from inside a Jupyter cell.
    args, _unknown = parser.parse_known_args()  # FIX: parse_args() -> parse_known_args()

    run_qc_batch(subjects=args.subjects, out=args.out, raw_path=args.raw, clean_path=args.clean)

No subjects given -- auto-discovered 26 subjects under E:\kntu\Thesis\codes\DATA: ['VP001', 'VP002', 'VP003', 'VP004', 'VP005', 'VP006', 'VP007', 'VP008', 'VP009', 'VP010', 'VP011', 'VP012', 'VP013', 'VP014', 'VP015', 'VP016', 'VP017', 'VP018', 'VP019', 'VP020', 'VP021', 'VP022', 'VP023', 'VP024', 'VP025', 'VP026']

QC: VP001
Running amplitude/flatline/clipping/kurtosis checks...
Running line-noise (50/60 Hz) detection...
Computing band power and paired statistical tests (Wilcoxon / paired t-test)...
Running stationarity test (Augmented Dickey-Fuller)...
Running normality tests (Shapiro-Wilk / Kolmogorov-Smirnov)...
Computing inter-channel correlation...
Computing Hjorth parameters...
Computing broadband SNR improvement...
Running ICA-based residual-artifact classification on preprocessed data...
Generating plots...

EEG PREPROCESSING QC REPORT
[FAIL]  RAW: fraction of gross-amplitude epochs (>150.0 uV p2p, 2s epochs)
        raw=0.3321754233608337  clean=None
        -> Likely artifac

In [ ]:
#VP004

In [2]:
"""
EEG Preprocessing Quality-Control (QC) Diagnostic Suite
=========================================================

Purpose
-------
Given a RAW (unprocessed) EEG recording and its PREPROCESSED counterpart (same
subject, same recording -- e.g. output of a notch+bandpass+ICA+re-reference
pipeline), this script runs a battery of standard, literature-established
statistical tests and diagnostic plots to answer two questions:

  1) Did preprocessing actually work (is the cleaned signal statistically
     "better" than the raw one on every axis that matters)?
  2) If something still looks wrong, WHICH artifact/noise source is most
     likely responsible (line noise, eye blinks/EOG, muscle/EMG, heartbeat/ECG,
     electrode drift/flat channels, motion, bad channels, non-stationarity)?

Input
-----
Two file paths pointing to EEG recordings MNE can read (.fif, .edf, .bdf, .set,
.vhdr, or a .mat/.npy handled by the loader below), OR two already-loaded
mne.io.Raw objects. See `load_raw()`.

Output
------
- A structured, printed QC report (per-metric PASS/WARN/FAIL with the specific
  artifact it implicates).
- A `results` dict (JSON-serializable) with every computed metric.
- A multi-panel PNG figure per diagnostic category (PSD, topomap, ICA,
  stationarity, correlation, Hjorth, per-channel boxplots).

Dependencies
------------
mne, mne_icalabel (optional, for automatic IC classification), numpy, scipy,
matplotlib, seaborn, statsmodels (for the ADF stationarity test).

Usage
-----
    python eeg_preprocessing_qc.py --raw path/to/raw.fif --clean path/to/clean.fif \
        --out ./qc_report

Or import and call `run_qc(raw, clean, out_dir="qc_report")` from your own code.
"""

import os
import json
import warnings
import argparse
from dataclasses import dataclass, field

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import mne
import scipy.io as sio
from scipy import stats, signal
from scipy.stats import kurtosis, skew, shapiro, kstest, wilcoxon, ttest_rel, entropy as scipy_entropy

try:
    from statsmodels.tsa.stattools import adfuller
    HAS_STATSMODELS = True
except ImportError:
    HAS_STATSMODELS = False

try:
    import mne_icalabel
    HAS_ICALABEL = True
except ImportError:
    HAS_ICALABEL = False

try:
    import seaborn as sns
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------------------
# CONFIGURATION -- thresholds are literature-typical defaults; tune per dataset
# ----------------------------------------------------------------------------

LINE_FREQS = [50.0, 60.0]           # check both in case region is unknown
LINE_BAND_HALFWIDTH = 1.0           # Hz, window around the line frequency
LINE_NOISE_SNR_DB_THRESHOLD = 6.0   # dB peak-to-neighbourhood; above => line noise present

EEG_BANDS = {
    "delta": (0.5, 4.0),
    "theta": (4.0, 8.0),
    "alpha": (8.0, 13.0),
    "beta":  (13.0, 30.0),
    "gamma": (30.0, 45.0),
    "muscle_hf": (45.0, 100.0),     # EMG/muscle artifact typically dominates here
}

FLATLINE_STD_UV_THRESHOLD = 0.05    # microvolt std below this => dead/flat channel
CLIPPING_SATURATION_FRACTION = 0.01 # fraction of samples at min/max => clipping
AMPLITUDE_REJECT_UV = 150.0         # generic "gross artifact" peak-to-peak threshold
KURTOSIS_ARTIFACT_THRESHOLD = 5.0   # excess kurtosis above this suggests spiky artifacts (eye blinks/EMG)
CORR_LOW_THRESHOLD = 0.4            # inter-channel correlation below this (median) suggests bad channel/noise
ADF_PVALUE_THRESHOLD = 0.05         # ADF test: p < 0.05 => stationary (good)
SHAPIRO_ALPHA = 0.05                # normality test alpha (informational, EEG is rarely perfectly Gaussian)

# ----------------------------------------------------------------------------
# DATASET-SPECIFIC CONFIG (Shin et al. 2018 n-back, same layout/format as your
# preprocessing script). Update these two paths so they match your
# preprocessing script's BASE_DATASET_PATH / OUTPUT_PATH exactly -- everything
# else (subject folder naming, file names) is copied verbatim from your
# load_eeg_data() / process_subject() so the QC script reads the SAME raw
# files and the SAME preprocessed outputs your pipeline produces, with no
# manual path-typing needed per subject.
# ----------------------------------------------------------------------------
BASE_DATASET_PATH = r"E:\kntu\Thesis\codes\DATA"                              # NEW: same as preprocessing script's BASE_DATASET_PATH
OUTPUT_PATH       = r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"   # NEW: same as preprocessing script's OUTPUT_PATH
SFREQ_EEG         = 200.0                                                     # NEW: same as preprocessing script's SFREQ_EEG
EEG_BAND_ORDER     = ["delta", "theta", "alpha", "beta", "gamma"]             # NEW: must match preprocessing script's EEG_BAND_ORDER


def build_real_eeg_montage(subject_id, ch_names):
    """
    FIX (per user correction): the dataset provides REAL electrode 3D positions in
    BASE_DATASET_PATH\\{subject_id}-EEG\\mnt_nback.mat (fields .pos_3d shape (3,30), .clab shape
    (30,)) -- this is the SAME file your preprocessing script's get_eeg_mnt() reads for
    spatial_mapping(). The generic 'standard_1005' template used previously is a guessed template
    montage, NOT this dataset's real electrode layout, and should not be used for topomap/ICA
    plotting when the real positions are available.

    This builds a proper mne.channels.DigMontage from mnt_nback.mat, matched BY NAME to `ch_names`
    (same name-matching approach as get_eeg_mnt() in your preprocessing script, since channel order
    in cnt_nback.mat vs mnt_nback.mat is not guaranteed identical). Returns None (caller should
    skip set_montage) if mnt_nback.mat is missing or no channels match.
    """
    eeg_folder = os.path.join(BASE_DATASET_PATH, f"{subject_id}-EEG")
    mnt_path = os.path.join(eeg_folder, "mnt_nback.mat")
    if not os.path.exists(mnt_path):
        print(f"  WARNING: {mnt_path} not found -- cannot build a real montage for {subject_id}, "
              f"topomap/ICA plots will be skipped.")
        return None

    mat_mnt = sio.loadmat(mnt_path, struct_as_record=False, squeeze_me=True)
    real_mnt_keys = [k for k in mat_mnt.keys() if not k.startswith("__")]
    mnt_struct = mat_mnt[real_mnt_keys[0]]

    full_clab = [str(c).strip() for c in np.asarray(mnt_struct.clab).flatten()]
    full_pos_3d = np.asarray(mnt_struct.pos_3d, dtype=np.float64)  # (3, n_channels_in_mnt)
    name_to_idx = {name: i for i, name in enumerate(full_clab)}
    name_to_idx_upper = {name.upper(): i for i, name in enumerate(full_clab)}

    ch_pos = {}
    for ch in ch_names:
        idx = name_to_idx.get(ch, name_to_idx_upper.get(ch.upper(), None))
        if idx is not None:
            ch_pos[ch] = full_pos_3d[:, idx]

    if not ch_pos:
        print(f"  WARNING: no channel names in {ch_names} matched mnt_nback.mat's clab for "
              f"{subject_id} -- topomap/ICA plots will be skipped.")
        return None

    montage = mne.channels.make_dig_montage(ch_pos=ch_pos, coord_frame="head")
    return montage


def load_raw_from_dataset(subject_id):
    """
    Loads the RAW (unprocessed) continuous EEG for `subject_id` (e.g. "VP001")
    directly from BASE_DATASET_PATH\\{subject_id}-EEG\\cnt_nback.mat, using the
    SAME reading logic as load_eeg_data() in your preprocessing script (same
    struct fields, same unit auto-detection, same channel typing). Returns an
    mne.io.Raw object -- no filtering/ICA/re-referencing applied (this is the
    RAW arm of the comparison).
    """
    eeg_folder = os.path.join(BASE_DATASET_PATH, f"{subject_id}-EEG")
    cnt_path = os.path.join(eeg_folder, "cnt_nback.mat")
    if not os.path.exists(cnt_path):
        raise FileNotFoundError(f"Raw EEG file not found: {cnt_path}")

    mat_cnt = sio.loadmat(cnt_path, struct_as_record=False, squeeze_me=True)
    real_cnt_keys = [k for k in mat_cnt.keys() if not k.startswith("__")]
    cnt_struct = mat_cnt[real_cnt_keys[0]]

    data = cnt_struct.x.T
    sfreq = float(cnt_struct.fs)
    ch_names = [str(ch).strip() for ch in cnt_struct.clab]

    if np.max(np.abs(data)) > 1.0:
        data = data * 1e-6  # match preprocessing script's uV -> V auto-detection

    if sfreq != SFREQ_EEG:
        n_samples = int(data.shape[1] * SFREQ_EEG / sfreq)
        data = signal.resample(data, n_samples, axis=1)
        sfreq = SFREQ_EEG

    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types="eeg")
    raw = mne.io.RawArray(data, info, verbose=False)
    eog_channels = [ch for ch in ["VEOG", "HEOG"] if ch in ch_names]
    if eog_channels:
        raw.set_channel_types({ch: "eog" for ch in eog_channels})
    # FIX (per user correction): use REAL electrode positions from this subject's own
    # mnt_nback.mat (same source your preprocessing script's spatial_mapping() uses), not the
    # generic 'standard_1005' template montage used previously.
    montage = build_real_eeg_montage(subject_id, raw.ch_names)  # FIX
    if montage is not None:  # FIX
        raw.set_montage(montage, match_case=False, on_missing="ignore")  # FIX
    return raw


def load_clean_from_pipeline(subject_id):
    """
    Loads the PREPROCESSED continuous EEG for `subject_id` from your pipeline's
    OUTPUT_PATH\\{subject_id}\\ folder. Two supported sources, tried in order:

    1) PRIMARY / RECOMMENDED: a continuous, channel-space, post-ICA .fif file
       named "{subject_id}_eeg_clean_raw.fif". Your preprocessing script does
       NOT currently save this -- add ONE line to process_subject(), right
       after `raw_eeg_clean = preprocess_eeg(raw_eeg, events)`:

           raw_eeg_clean.save(os.path.join(sub_out_dir, f"{subject_id}_eeg_clean_raw.fif"),
                               overwrite=True)

       (sub_out_dir is created a few lines below that point in your script;
       if you add the save line before sub_out_dir exists, move the
       os.makedirs(sub_out_dir, exist_ok=True) line up, or just os.makedirs
       here too.) This gives an exact, non-lossy comparison against raw.

    2) FALLBACK (approximate): if the .fif above doesn't exist, this function
       reconstructs a pseudo-continuous signal from the saved windowed,
       5-band, spatially-mapped tensor "{subject_id}_eeg_windows.npy"
       (shape (N,16,16,5,600)) by averaging over the 16x16 spatial grid to
       get back to a coarse "virtual channel" per band, then concatenating
       windows along time. This is LOSSY (spatial detail is destroyed by the
       averaging, and windows are concatenated with their 66% overlap simply
       ignored) and is only good enough for spectral/statistical sanity
       checks -- NOT a substitute for source (1) above. A clear WARNING is
       printed when this fallback is used.
    """
    sub_dir = os.path.join(OUTPUT_PATH, subject_id)

    fif_path = os.path.join(sub_dir, f"{subject_id}_eeg_clean_raw.fif")
    if os.path.exists(fif_path):
        clean_raw = mne.io.read_raw_fif(fif_path, preload=True, verbose=False)
        # FIX (per user correction): raw_eeg_clean in your preprocessing pipeline was originally
        # built from load_eeg_data(), which sets the generic 'standard_1005' template montage (kept
        # there only for interpolate_bads()'s internal use, per your pipeline's own comments) -- NOT
        # this subject's real electrode positions. Re-apply the REAL montage from mnt_nback.mat here
        # so topomap/ICA plotting on the preprocessed data uses actual electrode geometry, exactly
        # matching what spatial_mapping()/get_eeg_mnt() use in your preprocessing script.
        montage = build_real_eeg_montage(subject_id, clean_raw.ch_names)  # FIX
        if montage is not None:  # FIX
            clean_raw.set_montage(montage, match_case=False, on_missing="ignore")  # FIX
        return clean_raw

    windows_path = os.path.join(sub_dir, f"{subject_id}_eeg_windows.npy")
    if not os.path.exists(windows_path):
        raise FileNotFoundError(
            f"Neither '{fif_path}' nor '{windows_path}' found for {subject_id}. "
            f"Check OUTPUT_PATH and that preprocessing has been run for this subject."
        )

    print(f"  WARNING: '{subject_id}_eeg_clean_raw.fif' not found -- falling back to an "
          f"APPROXIMATE reconstruction from windowed data ({windows_path}). "
          f"See load_clean_from_pipeline() docstring to enable the exact comparison instead.")

    windows = np.load(windows_path)  # (N, 16, 16, 5, 600)
    n_windows, gH, gW, n_bands, n_times = windows.shape
    # Average over the spatial grid -> one pseudo-channel per band, then treat each band as a
    # "channel" and concatenate windows along time (ignoring overlap) to get a long pseudo-signal.
    pseudo = windows.mean(axis=(1, 2))               # (N, n_bands, n_times)
    pseudo = np.transpose(pseudo, (1, 0, 2))          # (n_bands, N, n_times)
    pseudo = pseudo.reshape(n_bands, n_windows * n_times)  # (n_bands, N*n_times)

    ch_names = [f"pseudo_{b}" for b in EEG_BAND_ORDER[:n_bands]]
    info = mne.create_info(ch_names=ch_names, sfreq=SFREQ_EEG, ch_types="eeg")
    return mne.io.RawArray(pseudo.astype(np.float64), info, verbose=False)


# ----------------------------------------------------------------------------
# DATA STRUCTURES
# ----------------------------------------------------------------------------

@dataclass
class QCResult:
    metric: str
    raw_value: float = None
    clean_value: float = None
    status: str = "INFO"           # PASS / WARN / FAIL / INFO
    likely_artifact: str = ""
    detail: str = ""

    def to_dict(self):
        return dict(metric=self.metric, raw_value=self.raw_value,
                     clean_value=self.clean_value, status=self.status,
                     likely_artifact=self.likely_artifact, detail=self.detail)


# ----------------------------------------------------------------------------
# I/O
# ----------------------------------------------------------------------------

def load_raw(path_or_raw, sfreq_if_array=250.0, ch_names=None):
    """
    Load an EEG recording into an mne.io.Raw object.
    Accepts: a path (.fif/.edf/.bdf/.set/.vhdr), an mne Raw object already in
    memory, or a numpy array of shape (n_channels, n_times) (in which case a
    generic EEG montage is created).
    """
    if isinstance(path_or_raw, mne.io.BaseRaw):
        return path_or_raw

    if isinstance(path_or_raw, np.ndarray):
        n_ch = path_or_raw.shape[0]
        names = ch_names or [f"EEG{i+1:03d}" for i in range(n_ch)]
        info = mne.create_info(names, sfreq_if_array, ch_types="eeg")
        return mne.io.RawArray(path_or_raw, info, verbose=False)

    ext = os.path.splitext(path_or_raw)[1].lower()
    if ext == ".fif":
        return mne.io.read_raw_fif(path_or_raw, preload=True, verbose=False)
    elif ext in (".edf",):
        return mne.io.read_raw_edf(path_or_raw, preload=True, verbose=False)
    elif ext in (".bdf",):
        return mne.io.read_raw_bdf(path_or_raw, preload=True, verbose=False)
    elif ext == ".set":
        return mne.io.read_raw_eeglab(path_or_raw, preload=True, verbose=False)
    elif ext == ".vhdr":
        return mne.io.read_raw_brainvision(path_or_raw, preload=True, verbose=False)
    elif ext == ".npy":
        arr = np.load(path_or_raw)
        return load_raw(arr, sfreq_if_array=sfreq_if_array, ch_names=ch_names)
    else:
        raise ValueError(f"Unsupported file extension: {ext}")


def _get_data_uv(raw):
    """Return data in microvolts, shape (n_channels, n_times)."""
    data = raw.get_data(picks="eeg")
    # MNE stores EEG in volts; heuristically detect and convert to microvolts.
    if np.median(np.abs(data)) < 1e-3:
        data = data * 1e6
    return data


# ----------------------------------------------------------------------------
# 1) BASIC SIGNAL-QUALITY METRICS (amplitude / flatline / clipping / kurtosis)
# ----------------------------------------------------------------------------

def check_amplitude_and_flatline(raw, label, results, epoch_seconds=2.0):
    data = _get_data_uv(raw)
    ch_names = raw.ch_names[:data.shape[0]]

    stds = data.std(axis=1)
    ptp = np.ptp(data, axis=1)  # whole-recording per-channel ptp -- kept for the flat/clip checks below,
    # which legitimately want a whole-recording view (a channel is "dead" or "saturated" as a whole).
    kurt = kurtosis(data, axis=1, fisher=True)  # excess kurtosis; ~0 for Gaussian, high for spiky artifacts
    skewness = skew(data, axis=1)

    flat_channels = [ch_names[i] for i, s in enumerate(stds) if s < FLATLINE_STD_UV_THRESHOLD]
    high_kurt_channels = [ch_names[i] for i, k in enumerate(kurt) if k > KURTOSIS_ARTIFACT_THRESHOLD]

    # clipping / saturation: fraction of samples at the extreme min/max of each channel
    clip_fracs = []
    for i in range(data.shape[0]):
        ch = data[i]
        lo, hi = ch.min(), ch.max()
        frac = np.mean((ch <= lo + 1e-9) | (ch >= hi - 1e-9))
        clip_fracs.append(frac)
    clipped_channels = [ch_names[i] for i, f in enumerate(clip_fracs) if f > CLIPPING_SATURATION_FRACTION]

    # FIX (per chat history -- real bug, not a preprocessing issue): the previous version computed
    # np.ptp(data, axis=1), which is the peak-to-peak of each channel over the ENTIRE continuous
    # recording, then averaged across CHANNELS -- despite being labelled "fraction of gross-
    # amplitude EPOCHS". A single transient (one imperfectly-removed blink, one brief movement)
    # anywhere in a multi-minute recording makes that channel's whole-recording ptp exceed the
    # threshold, so this metric was almost guaranteed to read high even for well-cleaned data --
    # it never actually measured what fraction of short time windows are contaminated, which is
    # what "epoch rejection" means in the EEG literature (e.g. the fixed-length epoching used by
    # autoreject / FASTER-style pipelines). Genuine epoch-based rejection below: the continuous
    # signal is cut into non-overlapping `epoch_seconds`-long windows, ptp is computed per
    # (epoch, channel), and the fraction of epoch-channel cells exceeding the threshold is reported
    # -- this is the metric the label always claimed to be.
    sfreq = raw.info["sfreq"]
    n_per_epoch = int(epoch_seconds * sfreq)
    n_epochs = data.shape[1] // n_per_epoch
    if n_epochs >= 1:
        epoched = data[:, :n_epochs * n_per_epoch].reshape(data.shape[0], n_epochs, n_per_epoch)
        epoch_ptp = np.ptp(epoched, axis=2)  # (n_channels, n_epochs)
        gross_artifact_frac = np.mean(epoch_ptp > AMPLITUDE_REJECT_UV)  # FIX: real epoch-channel fraction
    else:
        gross_artifact_frac = float(np.mean(ptp > AMPLITUDE_REJECT_UV))  # fallback for very short signals

    results.append(QCResult(
        metric=f"{label}: flat/dead channels (std < {FLATLINE_STD_UV_THRESHOLD} uV)",
        raw_value=len(flat_channels), status="FAIL" if flat_channels else "PASS",
        likely_artifact="Electrode disconnection / dead channel" if flat_channels else "",
        detail=str(flat_channels)))

    results.append(QCResult(
        metric=f"{label}: clipped/saturated channels (>{CLIPPING_SATURATION_FRACTION*100:.1f}% samples at extremes)",
        raw_value=len(clipped_channels), status="FAIL" if clipped_channels else "PASS",
        likely_artifact="Amplifier saturation / electrode-skin contact failure" if clipped_channels else "",
        detail=str(clipped_channels)))

    results.append(QCResult(
        metric=f"{label}: high-kurtosis channels (excess kurtosis > {KURTOSIS_ARTIFACT_THRESHOLD})",
        raw_value=len(high_kurt_channels), status="WARN" if high_kurt_channels else "PASS",
        likely_artifact="Spiky transient artifacts (eye blinks, electrode pops, EMG bursts)" if high_kurt_channels else "",
        detail=str(high_kurt_channels)))

    results.append(QCResult(
        metric=f"{label}: fraction of gross-amplitude epochs (>{AMPLITUDE_REJECT_UV} uV p2p, "
               f"{epoch_seconds:.0f}s epochs)",  # FIX: label now matches what's actually computed
        raw_value=float(gross_artifact_frac),
        status="FAIL" if gross_artifact_frac > 0.10 else ("WARN" if gross_artifact_frac > 0.02 else "PASS"),
        likely_artifact="Movement artifact / muscle burst / electrode pop" if gross_artifact_frac > 0.02 else "",
        detail=f"mean std={stds.mean():.2f}uV, mean whole-recording ptp={ptp.mean():.2f}uV, "
               f"mean skew={skewness.mean():.2f}"))

    return dict(stds=stds, ptp=ptp, kurt=kurt, skew=skewness, clip_fracs=np.array(clip_fracs))


# ----------------------------------------------------------------------------
# 2) LINE NOISE (50/60 Hz) DETECTION -- Welch PSD peak test
# ----------------------------------------------------------------------------

def check_line_noise(raw, label, results):
    data = _get_data_uv(raw)
    sfreq = raw.info["sfreq"]
    freqs, psd = signal.welch(data, fs=sfreq, nperseg=min(int(4 * sfreq), data.shape[1]), axis=1)
    psd_db = 10 * np.log10(psd.mean(axis=0) + 1e-20)

    # FIX (per chat history -- real bug, not a preprocessing issue): if `raw` has already been
    # lowpass-filtered (MNE records this in raw.info['lowpass'] whenever .filter(h_freq=...) was
    # called), checking for line noise at a frequency ABOVE that lowpass cutoff means comparing two
    # regions that are BOTH deep in the filter's stopband -- i.e. both at the numerical attenuation
    # floor of the FIR filter, not at any real signal power. Any tiny difference in the filter's
    # roll-off shape between the "peak" window (49-51 Hz) and "neighbour" windows (45-48/52-55 Hz)
    # then reads as a spurious multi-dB "SNR", uniformly across every subject and, tellingly, at
    # BOTH 50 Hz AND 60 Hz roughly equally (both sit in the same already-attenuated region above a
    # 40 Hz cutoff) -- which is exactly the pattern observed in your VP001-VP011 results and is the
    # signature of a stopband-floor artifact, not genuine residual mains interference. This check
    # is only meaningful for signals whose passband still covers the probed frequency.
    lowpass = raw.info.get("lowpass", None)  # NEW

    line_hits = {}
    skipped_stopband = []  # NEW
    for lf in LINE_FREQS:
        if lf >= sfreq / 2:
            continue
        if lowpass is not None and (lf - LINE_BAND_HALFWIDTH) > lowpass:  # NEW
            skipped_stopband.append(lf)  # NEW: entire probe band lies above the filter's passband
            continue  # NEW
        band_mask = (freqs >= lf - LINE_BAND_HALFWIDTH) & (freqs <= lf + LINE_BAND_HALFWIDTH)
        neigh_mask = ((freqs >= lf - 5) & (freqs <= lf - 2)) | ((freqs >= lf + 2) & (freqs <= lf + 5))
        if not band_mask.any() or not neigh_mask.any():
            continue
        peak_db = psd_db[band_mask].max()
        neigh_db = psd_db[neigh_mask].mean()
        snr_db = peak_db - neigh_db
        line_hits[lf] = snr_db

    if not line_hits and skipped_stopband:  # NEW: every line frequency was above the filter's passband
        results.append(QCResult(  # NEW
            metric=f"{label}: line noise peak SNR (50/60 Hz)",  # NEW
            status="INFO",  # NEW
            detail=f"Skipped -- {skipped_stopband} Hz all lie above this signal's lowpass filter "  # NEW
                   f"cutoff ({lowpass} Hz), so any 'line noise' reading there would just be "  # NEW
                   f"filter stopband floor, not real interference."))  # NEW
        return freqs, psd_db  # NEW

    worst_freq, worst_snr = (max(line_hits.items(), key=lambda kv: kv[1])
                              if line_hits else (None, -np.inf))
    status = "FAIL" if worst_snr > LINE_NOISE_SNR_DB_THRESHOLD else "PASS"

    results.append(QCResult(
        metric=f"{label}: line noise peak SNR (50/60 Hz)",
        raw_value=float(worst_snr) if worst_snr != -np.inf else None,
        status=status,
        likely_artifact=f"Power-line interference ({worst_freq} Hz)" if status == "FAIL" else "",
        detail=str({k: round(v, 2) for k, v in line_hits.items()}) +
               (f" (skipped {skipped_stopband} Hz -- above lowpass cutoff {lowpass} Hz)" if skipped_stopband else "")))  # NEW

    return freqs, psd_db


# ----------------------------------------------------------------------------
# 3) BAND-POWER / SPECTRAL PROFILE + PAIRED STATISTICAL TESTS (raw vs clean)
# ----------------------------------------------------------------------------

def compute_band_power(raw):
    data = _get_data_uv(raw)
    sfreq = raw.info["sfreq"]
    freqs, psd = signal.welch(data, fs=sfreq, nperseg=min(int(4 * sfreq), data.shape[1]), axis=1)
    band_power = {}
    for band, (lo, hi) in EEG_BANDS.items():
        mask = (freqs >= lo) & (freqs <= hi)
        band_power[band] = psd[:, mask].mean(axis=1)  # per-channel mean power in band
    return band_power, freqs, psd


def compare_band_power(raw, clean, results):
    bp_raw, freqs_r, psd_r = compute_band_power(raw)
    bp_clean, freqs_c, psd_c = compute_band_power(clean)

    for band in EEG_BANDS:
        a = bp_raw[band]
        b = bp_clean[band]
        n = min(len(a), len(b))
        a, b = a[:n], b[:n]

        # Paired statistical test: Wilcoxon signed-rank (non-parametric, robust to
        # non-Gaussian EEG power distributions) is the primary test; paired t-test
        # reported alongside as the classical parametric counterpart.
        try:
            wilcoxon_stat, wilcoxon_p = wilcoxon(a, b)
        except ValueError:
            wilcoxon_stat, wilcoxon_p = np.nan, np.nan
        ttest_stat, ttest_p = ttest_rel(a, b)

        pct_change = 100 * (b.mean() - a.mean()) / (a.mean() + 1e-20)

        artifact_hint = ""
        if band == "muscle_hf" and pct_change < -20:
            artifact_hint = "EMG/muscle power reduced by cleaning (expected & good)"
        elif band == "muscle_hf" and pct_change > 20:
            artifact_hint = "Residual EMG/muscle artifact NOT removed by preprocessing"
        elif band in ("delta",) and pct_change > 50:
            artifact_hint = "Possible residual slow drift / sweat artifact in delta band"

        results.append(QCResult(
            metric=f"Band power [{band}] raw vs clean (Wilcoxon signed-rank)",
            raw_value=float(a.mean()), clean_value=float(b.mean()),
            status="INFO",
            likely_artifact=artifact_hint,
            detail=f"pct_change={pct_change:+.1f}%, wilcoxon_p={wilcoxon_p:.4g}, paired_t_p={ttest_p:.4g}"))

    return bp_raw, bp_clean, freqs_r, psd_r, freqs_c, psd_c


# ----------------------------------------------------------------------------
# 4) STATIONARITY -- Augmented Dickey-Fuller test (classic econometrics/signal
#    processing test; non-stationary EEG segments often indicate drift/motion)
# ----------------------------------------------------------------------------

def check_stationarity(raw, label, results, max_channels=8):
    if not HAS_STATSMODELS:
        results.append(QCResult(metric=f"{label}: stationarity (ADF test)", status="INFO",
                                 detail="statsmodels not installed -- skipped"))
        return None

    data = _get_data_uv(raw)
    n_ch = min(max_channels, data.shape[0])
    pvals = []
    for i in range(n_ch):
        # subsample for speed on long recordings
        seg = data[i, ::max(1, data.shape[1] // 20000)]
        try:
            adf_stat, pvalue, *_ = adfuller(seg, autolag="AIC")
        except Exception:
            pvalue = np.nan
        pvals.append(pvalue)
    pvals = np.array(pvals)
    frac_nonstationary = np.mean(pvals >= ADF_PVALUE_THRESHOLD)

    results.append(QCResult(
        metric=f"{label}: fraction of non-stationary channels (ADF p >= {ADF_PVALUE_THRESHOLD})",
        raw_value=float(frac_nonstationary),
        status="WARN" if frac_nonstationary > 0.3 else "PASS",
        likely_artifact="Slow drift / baseline wander / motion artifact" if frac_nonstationary > 0.3 else "",
        detail=f"mean ADF p-value={np.nanmean(pvals):.4g}"))
    return pvals


# ----------------------------------------------------------------------------
# 5) NORMALITY -- Shapiro-Wilk / Kolmogorov-Smirnov (informational: EEG amplitude
#    distributions deviating strongly from Gaussian often indicate artifact
#    contamination, since clean cortical EEG is approximately Gaussian at rest)
# ----------------------------------------------------------------------------

def check_normality(raw, label, results, max_channels=8, max_samples=5000):
    data = _get_data_uv(raw)
    n_ch = min(max_channels, data.shape[0])
    sw_pvals, ks_pvals = [], []
    for i in range(n_ch):
        seg = data[i]
        if len(seg) > max_samples:
            idx = np.random.choice(len(seg), max_samples, replace=False)
            seg = seg[idx]
        try:
            _, p_sw = shapiro(seg)
        except Exception:
            p_sw = np.nan
        _, p_ks = kstest((seg - seg.mean()) / (seg.std() + 1e-12), "norm")
        sw_pvals.append(p_sw)
        ks_pvals.append(p_ks)

    frac_nonnormal = np.mean(np.array(sw_pvals) < SHAPIRO_ALPHA)
    results.append(QCResult(
        metric=f"{label}: fraction of non-Gaussian channels (Shapiro-Wilk p < {SHAPIRO_ALPHA})",
        raw_value=float(frac_nonnormal), status="INFO",
        likely_artifact="Heavy-tailed/non-Gaussian amplitude distribution -- consistent with residual "
                         "blink/muscle spikes if fraction is high" if frac_nonnormal > 0.5 else "",
        detail=f"mean Shapiro p={np.nanmean(sw_pvals):.4g}, mean KS p={np.nanmean(ks_pvals):.4g}"))
    return sw_pvals, ks_pvals


# ----------------------------------------------------------------------------
# 6) INTER-CHANNEL CORRELATION -- low correlation flags noisy/bad channels;
#    abnormally HIGH uniform correlation across all channels flags a common
#    artifact source (e.g. unremoved eye-blink component riding on all channels)
# ----------------------------------------------------------------------------

def check_channel_correlation(raw, label, results):
    data = _get_data_uv(raw)
    n_ch = data.shape[0]
    corr = np.corrcoef(data)
    np.fill_diagonal(corr, np.nan)
    mean_corr_per_ch = np.nanmean(corr, axis=1)
    global_mean_corr = np.nanmean(corr)

    # FIX (per chat history -- real bug, not a preprocessing issue): common-average referencing
    # (CAR) forces sum_i x_i(t) = 0 at every timepoint, which mathematically drives the EXPECTED
    # mean pairwise correlation toward -1/(N-1) (e.g. -1/27 ~= -0.037 for 28 channels) even for a
    # perfectly healthy, well-preprocessed recording -- this is a property of the referencing
    # scheme, not evidence of noise. The original CORR_LOW_THRESHOLD=0.4 was calibrated for
    # un-referenced (or non-CAR) data and, when applied to CAR'd data, flagged nearly every channel
    # as "low correlation" in every subject -- exactly the uniform, all-channels-flagged pattern
    # seen across VP001-VP011. FIX: detect whether CAR was applied (via raw.info['custom_ref_applied']
    # if the .fif carries it, with a name-based EEG count fallback) and shift the low-correlation
    # threshold down by the expected CAR baseline instead of using a fixed absolute cutoff.
    is_car = bool(raw.info.get("custom_ref_applied", False))  # FIX: simplified, confirmed mne.Info.get() works directly
    expected_baseline = -1.0 / max(n_ch - 1, 1) if (is_car or n_ch > 1) else 0.0  # NEW
    # FIX (bug in the first version of this fix): "CORR_LOW_THRESHOLD - 0.4" always evaluated to
    # exactly 0 (since CORR_LOW_THRESHOLD's own default value IS 0.4), which put the effective
    # threshold EXACTLY AT the CAR baseline with zero tolerance -- flagging ~half of all channels
    # purely from random noise scattering around that baseline (confirmed by test: 12/28 channels
    # flagged on a fully synthetic, artifact-free signal). Use an explicit, independent margin
    # instead: a channel is flagged only if its correlation falls meaningfully BELOW the
    # theoretical CAR baseline, not merely at or slightly below it.
    CORR_MARGIN_BELOW_CAR_BASELINE = 0.15  # NEW: independent of CORR_LOW_THRESHOLD, tune if needed
    effective_threshold = (expected_baseline - CORR_MARGIN_BELOW_CAR_BASELINE) if is_car else CORR_LOW_THRESHOLD  # FIX

    low_corr_channels = [raw.ch_names[i] for i, c in enumerate(mean_corr_per_ch) if c < effective_threshold]  # CHANGED

    status = "WARN" if low_corr_channels else "PASS"
    artifact = ""
    if low_corr_channels:
        artifact = "Bad/noisy channel(s) poorly correlated with neighbours (possible electrode fault)"
    if global_mean_corr > 0.85:
        status = "WARN"
        artifact = (artifact + "; " if artifact else "") + \
            "Abnormally high global correlation -- possible unremoved common artifact " \
            "(eye blink / reference / movement affecting all channels similarly)"

    results.append(QCResult(
        metric=f"{label}: inter-channel correlation",
        raw_value=float(global_mean_corr), status=status,
        likely_artifact=artifact,
        detail=f"low-corr channels: {low_corr_channels} | "
               f"CAR-expected baseline: {expected_baseline:.3f}, effective threshold used: {effective_threshold:.3f}"))  # NEW
    return corr


# ----------------------------------------------------------------------------
# 7) HJORTH PARAMETERS (Activity/Mobility/Complexity) -- classic EEG QC metrics
# ----------------------------------------------------------------------------

def hjorth_parameters(data):
    """data: (n_channels, n_times). Returns activity, mobility, complexity per channel."""
    d1 = np.diff(data, axis=1)
    d2 = np.diff(d1, axis=1)
    var0 = np.var(data, axis=1)
    var1 = np.var(d1, axis=1)
    var2 = np.var(d2, axis=1)
    activity = var0
    mobility = np.sqrt(var1 / (var0 + 1e-20))
    complexity = np.sqrt(var2 / (var1 + 1e-20)) / (mobility + 1e-20)
    return activity, mobility, complexity


def check_hjorth(raw, label, results):
    data = _get_data_uv(raw)
    activity, mobility, complexity = hjorth_parameters(data)
    # Complexity >> 1.5-2 typically indicates irregular, artifact-like signal;
    # normal resting EEG complexity is roughly in the 1.0-1.5 range (dataset dependent).
    high_complexity_frac = np.mean(complexity > 2.0)
    results.append(QCResult(
        metric=f"{label}: Hjorth complexity (fraction of channels > 2.0)",
        raw_value=float(high_complexity_frac), status="INFO",
        likely_artifact="High signal irregularity -- consistent with EMG/muscle contamination" if high_complexity_frac > 0.3 else "",
        detail=f"mean activity={activity.mean():.2f}, mean mobility={mobility.mean():.3f}, "
               f"mean complexity={complexity.mean():.3f}"))
    return activity, mobility, complexity


# ----------------------------------------------------------------------------
# 8) SIGNAL-TO-NOISE RATIO IMPROVEMENT (raw vs clean) -- broadband RMS ratio in
#    the physiological band (1-40 Hz) vs the "noise-dominated" high band (>45 Hz)
# ----------------------------------------------------------------------------

def check_snr_improvement(raw, clean, results):
    def band_rms(raw_obj, lo, hi):
        d = raw_obj.copy().filter(lo, hi, method="fir", verbose=False).get_data(picks="eeg")
        return np.sqrt(np.mean(d ** 2))

    try:
        signal_rms_raw = band_rms(raw, 1.0, 40.0)
        noise_rms_raw = band_rms(raw, 45.0, min(99.0, raw.info["sfreq"] / 2 - 1))
        snr_raw = 20 * np.log10(signal_rms_raw / (noise_rms_raw + 1e-20))

        signal_rms_clean = band_rms(clean, 1.0, 40.0)
        noise_rms_clean = band_rms(clean, 45.0, min(99.0, clean.info["sfreq"] / 2 - 1))
        snr_clean = 20 * np.log10(signal_rms_clean / (noise_rms_clean + 1e-20))

        delta_snr = snr_clean - snr_raw
        results.append(QCResult(
            metric="Broadband SNR (1-40Hz signal / 45-100Hz noise band), dB",
            raw_value=float(snr_raw), clean_value=float(snr_clean),
            status="PASS" if delta_snr > 0 else "WARN",
            likely_artifact="Preprocessing did NOT improve SNR -- check filter settings / bad channels" if delta_snr <= 0 else "",
            detail=f"delta SNR = {delta_snr:+.2f} dB"))
    except Exception as e:
        results.append(QCResult(metric="Broadband SNR", status="INFO", detail=f"failed: {e}"))


# ----------------------------------------------------------------------------
# 9) ICA-BASED ARTIFACT SOURCE IDENTIFICATION (optional, needs mne_icalabel)
# ----------------------------------------------------------------------------

def check_ica_artifacts(clean, results, out_dir):
    if not HAS_ICALABEL:
        results.append(QCResult(metric="ICA component classification", status="INFO",
                                 detail="mne_icalabel not installed -- skipped"))
        return None

    try:
        clean_for_ica = clean.copy().filter(1.0, 40.0, verbose=False)
        clean_for_ica.set_eeg_reference("average", verbose=False)
        ica = mne.preprocessing.ICA(n_components=0.99, method="picard",
                                     fit_params=dict(extended=True), random_state=42, verbose=False)
        ica.fit(clean_for_ica, verbose=False)
        ic_labels = mne_icalabel.label_components(clean_for_ica, ica, method="iclabel")

        labels = ic_labels["labels"]
        probs = ic_labels["y_pred_proba"]
        counts = {}
        for lab in labels:
            counts[lab] = counts.get(lab, 0) + 1

        artifact_labels = {"eye blink", "muscle artifact", "heart beat", "line noise", "channel noise"}
        n_artifact = sum(v for k, v in counts.items() if k in artifact_labels)

        results.append(QCResult(
            metric="Residual ICA-detectable artifact components in PREPROCESSED data",
            raw_value=int(n_artifact), status="WARN" if n_artifact > 0 else "PASS",
            likely_artifact=", ".join(k for k in counts if k in artifact_labels and counts[k] > 0),
            detail=str(counts)))

        fig = ica.plot_components(show=False)
        figs = fig if isinstance(fig, list) else [fig]
        for i, f in enumerate(figs):
            f.savefig(os.path.join(out_dir, f"ica_components_{i}.png"), dpi=130)
            plt.close(f)

        return ic_labels
    except Exception as e:
        results.append(QCResult(metric="ICA component classification", status="INFO", detail=f"failed: {e}"))
        return None


# ----------------------------------------------------------------------------
# PLOTS
# ----------------------------------------------------------------------------

def plot_psd_comparison(freqs_r, psd_r, freqs_c, psd_c, out_dir):
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(freqs_r, psd_r, label="Raw", color="tab:red", alpha=0.8)
    ax.plot(freqs_c, psd_c, label="Preprocessed", color="tab:blue", alpha=0.8)
    for lf in LINE_FREQS:
        ax.axvline(lf, color="gray", linestyle="--", alpha=0.5)
    for band, (lo, hi) in EEG_BANDS.items():
        ax.axvspan(lo, hi, alpha=0.03)
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Power (dB, mean across channels)")
    ax.set_title("PSD: Raw vs Preprocessed (dashed = line-noise freqs, shaded = EEG bands)")
    ax.set_xlim(0, min(100, freqs_r.max()))
    ax.legend()
    plt.tight_layout()
    fig.savefig(os.path.join(out_dir, "psd_comparison.png"), dpi=150)
    plt.close(fig)


def plot_channel_boxplots(stats_raw, stats_clean, ch_names, out_dir):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    metrics = [("stds", "Std (uV)"), ("ptp", "Peak-to-peak (uV)"), ("kurt", "Excess kurtosis")]
    for ax, (key, title) in zip(axes, metrics):
        data_to_plot = [stats_raw[key], stats_clean[key]]
        ax.boxplot(data_to_plot, labels=["Raw", "Preprocessed"])
        ax.set_title(title)
    plt.tight_layout()
    fig.savefig(os.path.join(out_dir, "channel_quality_boxplots.png"), dpi=150)
    plt.close(fig)


def plot_correlation_heatmaps(corr_raw, corr_clean, out_dir):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, corr, title in zip(axes, [corr_raw, corr_clean], ["Raw", "Preprocessed"]):
        im = ax.imshow(np.nan_to_num(corr), vmin=-1, vmax=1, cmap="coolwarm")
        ax.set_title(f"Inter-channel correlation: {title}")
        plt.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout()
    fig.savefig(os.path.join(out_dir, "channel_correlation_heatmaps.png"), dpi=150)
    plt.close(fig)


def plot_topomap_band_power(clean, bp_clean, out_dir):
    try:
        eeg_picks = mne.pick_types(clean.info, eeg=True, eog=False)
        eeg_info = mne.pick_info(clean.info, eeg_picks)  # FIX: vals below only has one entry per EEG-type
        # channel (see compute_band_power's picks="eeg"), so the info object passed to plot_topomap
        # must be restricted the same way, or MNE raises a length-mismatch error whenever EOG/other
        # non-EEG channels are present in `clean` (e.g. VEOG/HEOG before they're dropped).
        n_bands = len(EEG_BANDS)
        fig, axes = plt.subplots(1, n_bands, figsize=(3 * n_bands, 3.5))
        for ax, (band, vals) in zip(axes, bp_clean.items()):
            mne.viz.plot_topomap(vals[:len(eeg_info["ch_names"])], eeg_info, axes=ax, show=False)  # FIX: eeg_info instead of clean.info
            ax.set_title(band)
        plt.tight_layout()
        fig.savefig(os.path.join(out_dir, "topomap_band_power.png"), dpi=150)
        plt.close(fig)
    except Exception as e:
        print(f"  [plot] topomap skipped (no valid montage / channel-position mismatch): {e}")


def plot_time_series_snippet(raw, clean, out_dir, duration=10.0, picks_n=5):
    sfreq = raw.info["sfreq"]
    n = int(duration * sfreq)
    data_r = _get_data_uv(raw)[:picks_n, :n]
    data_c = _get_data_uv(clean)[:picks_n, :min(n, _get_data_uv(clean).shape[1])]
    t_r = np.arange(data_r.shape[1]) / sfreq
    t_c = np.arange(data_c.shape[1]) / sfreq

    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
    offset = np.median(np.abs(data_r)) * 8 + 1e-6
    for i in range(data_r.shape[0]):
        axes[0].plot(t_r, data_r[i] + i * offset, linewidth=0.6, color="tab:red")
    axes[0].set_title(f"Raw (first {duration:.0f}s, {picks_n} channels)")
    for i in range(data_c.shape[0]):
        axes[1].plot(t_c, data_c[i] + i * offset, linewidth=0.6, color="tab:blue")
    axes[1].set_title(f"Preprocessed (first {duration:.0f}s, {picks_n} channels)")
    axes[1].set_xlabel("Time (s)")
    plt.tight_layout()
    fig.savefig(os.path.join(out_dir, "time_series_snippet.png"), dpi=150)
    plt.close(fig)


# ----------------------------------------------------------------------------
# REPORTING
# ----------------------------------------------------------------------------

def print_report(results):
    order = {"FAIL": 0, "WARN": 1, "INFO": 2, "PASS": 3}
    results_sorted = sorted(results, key=lambda r: order.get(r.status, 4))
    print("\n" + "=" * 100)
    print("EEG PREPROCESSING QC REPORT")
    print("=" * 100)
    for r in results_sorted:
        tag = f"[{r.status}]"
        print(f"{tag:<7} {r.metric}")
        if r.raw_value is not None or r.clean_value is not None:
            print(f"        raw={r.raw_value}  clean={r.clean_value}")
        if r.likely_artifact:
            print(f"        -> Likely artifact/cause: {r.likely_artifact}")
        if r.detail:
            print(f"        detail: {r.detail}")
        print("-" * 100)

    n_fail = sum(1 for r in results if r.status == "FAIL")
    n_warn = sum(1 for r in results if r.status == "WARN")
    print(f"\nSUMMARY: {n_fail} FAIL, {n_warn} WARN out of {len(results)} checks.")
    if n_fail == 0 and n_warn == 0:
        print("Preprocessing looks clean on all implemented checks.")
    elif n_fail > 0:
        print("Preprocessing has UNRESOLVED artifacts (see FAIL items above) -- do not proceed to modeling "
              "without addressing these.")
    else:
        print("Preprocessing is broadly acceptable but has minor residual issues (see WARN items).")


# ----------------------------------------------------------------------------
# MAIN ENTRY POINT
# ----------------------------------------------------------------------------

def run_qc(raw, clean, out_dir="qc_report"):
    os.makedirs(out_dir, exist_ok=True)
    raw = load_raw(raw)
    clean = load_raw(clean)

    results = []

    print("Running amplitude/flatline/clipping/kurtosis checks...")
    stats_raw = check_amplitude_and_flatline(raw, "RAW", results)
    stats_clean = check_amplitude_and_flatline(clean, "PREPROCESSED", results)

    print("Running line-noise (50/60 Hz) detection...")
    freqs_r, psd_r = check_line_noise(raw, "RAW", results)
    freqs_c, psd_c = check_line_noise(clean, "PREPROCESSED", results)

    print("Computing band power and paired statistical tests (Wilcoxon / paired t-test)...")
    bp_raw, bp_clean, freqs_r2, psd_r2, freqs_c2, psd_c2 = compare_band_power(raw, clean, results)

    print("Running stationarity test (Augmented Dickey-Fuller)...")
    check_stationarity(raw, "RAW", results)
    check_stationarity(clean, "PREPROCESSED", results)

    print("Running normality tests (Shapiro-Wilk / Kolmogorov-Smirnov)...")
    check_normality(raw, "RAW", results)
    check_normality(clean, "PREPROCESSED", results)

    print("Computing inter-channel correlation...")
    corr_raw = check_channel_correlation(raw, "RAW", results)
    corr_clean = check_channel_correlation(clean, "PREPROCESSED", results)

    print("Computing Hjorth parameters...")
    check_hjorth(raw, "RAW", results)
    check_hjorth(clean, "PREPROCESSED", results)

    print("Computing broadband SNR improvement...")
    check_snr_improvement(raw, clean, results)

    print("Running ICA-based residual-artifact classification on preprocessed data...")
    check_ica_artifacts(clean, results, out_dir)

    print("Generating plots...")
    plot_psd_comparison(freqs_r2, 10 * np.log10(psd_r2.mean(axis=0) + 1e-20),
                         freqs_c2, 10 * np.log10(psd_c2.mean(axis=0) + 1e-20), out_dir)
    plot_channel_boxplots(stats_raw, stats_clean, clean.ch_names, out_dir)
    plot_correlation_heatmaps(corr_raw, corr_clean, out_dir)
    plot_topomap_band_power(clean, bp_clean, out_dir)
    plot_time_series_snippet(raw, clean, out_dir)

    print_report(results)

    with open(os.path.join(out_dir, "qc_results.json"), "w") as f:
        json.dump([r.to_dict() for r in results], f, indent=2, default=str)

    print(f"\nAll plots and qc_results.json saved to: {os.path.abspath(out_dir)}")
    return results


def run_qc_for_subject(subject_id, out_root="qc_report"):
    """
    Convenience wrapper: loads RAW directly from BASE_DATASET_PATH and
    PREPROCESSED directly from OUTPUT_PATH for `subject_id` (e.g. "VP001"),
    using load_raw_from_dataset() / load_clean_from_pipeline() above, then
    runs the full QC suite and saves everything to out_root/{subject_id}/.
    """
    out_dir = os.path.join(out_root, subject_id)
    print(f"\n{'='*100}\nQC: {subject_id}\n{'='*100}")
    raw = load_raw_from_dataset(subject_id)
    clean = load_clean_from_pipeline(subject_id)
    return run_qc(raw, clean, out_dir=out_dir)


def run_qc_batch(subjects=None, out="qc_report", raw_path=None, clean_path=None):
    """
    Callable directly from a Jupyter cell (no argparse involved) -- this is what both the CLI
    branch below AND notebook users should call. Examples:

        # single manual raw/clean file pair:
        run_qc_batch(raw_path="C:/.../raw.fif", clean_path="C:/.../clean.fif", out="qc_report")

        # specific subjects from the dataset (uses BASE_DATASET_PATH / OUTPUT_PATH above):
        run_qc_batch(subjects=["VP001", "VP002"], out="qc_report")

        # every subject auto-discovered under BASE_DATASET_PATH:
        run_qc_batch(out="qc_report")
    """
    if raw_path and clean_path:
        return {"_manual_pair": [r.to_dict() for r in run_qc(raw_path, clean_path, out_dir=out)]}

    if subjects is None:
        subjects = sorted({
            name.split("-EEG")[0] for name in os.listdir(BASE_DATASET_PATH)
            if name.endswith("-EEG") and os.path.isdir(os.path.join(BASE_DATASET_PATH, name))
        })
        print(f"No subjects given -- auto-discovered {len(subjects)} subjects under "
              f"{BASE_DATASET_PATH}: {subjects}")

    all_results = {}
    for sid in subjects:
        try:
            all_results[sid] = [r.to_dict() for r in run_qc_for_subject(sid, out_root=out)]
        except Exception as e:
            print(f"  ERROR processing {sid}: {e}")

    os.makedirs(out, exist_ok=True)
    with open(os.path.join(out, "qc_all_subjects.json"), "w") as f:
        json.dump(all_results, f, indent=2, default=str)
    print(f"\nAll-subjects QC summary saved to: {os.path.join(out, 'qc_all_subjects.json')}")
    return all_results


print('eeg_preprocessing_qc.py function definitions loaded (no subjects processed).')

eeg_preprocessing_qc.py function definitions loaded (no subjects processed).


In [3]:
run_qc_batch(subjects=["VP004"], out=r"E:\kntu\Thesis\codes\HOPE-LAST-CODE-FILE\qc_report")


QC: VP004
Running amplitude/flatline/clipping/kurtosis checks...
Running line-noise (50/60 Hz) detection...
Computing band power and paired statistical tests (Wilcoxon / paired t-test)...
Running stationarity test (Augmented Dickey-Fuller)...
Running normality tests (Shapiro-Wilk / Kolmogorov-Smirnov)...
Computing inter-channel correlation...
Computing Hjorth parameters...
Computing broadband SNR improvement...
Running ICA-based residual-artifact classification on preprocessed data...
Generating plots...

EEG PREPROCESSING QC REPORT
[FAIL]  RAW: fraction of gross-amplitude epochs (>150.0 uV p2p, 2s epochs)
        raw=0.6509038800705468  clean=None
        -> Likely artifact/cause: Movement artifact / muscle burst / electrode pop
        detail: mean std=320.46uV, mean whole-recording ptp=2936.18uV, mean skew=-1.20
----------------------------------------------------------------------------------------------------
[FAIL]  PREPROCESSED: fraction of gross-amplitude epochs (>150.0 uV p2p

{'VP004': [{'metric': 'RAW: flat/dead channels (std < 0.05 uV)',
   'raw_value': 0,
   'clean_value': None,
   'status': 'PASS',
   'likely_artifact': '',
   'detail': '[]'},
  {'metric': 'RAW: clipped/saturated channels (>1.0% samples at extremes)',
   'raw_value': 0,
   'clean_value': None,
   'status': 'PASS',
   'likely_artifact': '',
   'detail': '[]'},
  {'metric': 'RAW: high-kurtosis channels (excess kurtosis > 5.0)',
   'raw_value': 0,
   'clean_value': None,
   'status': 'PASS',
   'likely_artifact': '',
   'detail': '[]'},
  {'metric': 'RAW: fraction of gross-amplitude epochs (>150.0 uV p2p, 2s epochs)',
   'raw_value': 0.6509038800705468,
   'clean_value': None,
   'status': 'FAIL',
   'likely_artifact': 'Movement artifact / muscle burst / electrode pop',
   'detail': 'mean std=320.46uV, mean whole-recording ptp=2936.18uV, mean skew=-1.20'},
  {'metric': 'PREPROCESSED: flat/dead channels (std < 0.05 uV)',
   'raw_value': 0,
   'clean_value': None,
   'status': 'PASS',
   'li

In [ ]:
#Kurtosis batch check all subjects

In [4]:
"""
SELF-CONTAINED CELL -- batch-runs the kurtosis-increase diagnostic (benign normalization effect
vs. genuine residual spikes) across EVERY subject whose QC report showed an INCREASE in
high-kurtosis channel count after preprocessing: VP004, VP005, VP006, VP007, VP008, VP010, VP011.

(VP001, VP003, VP009 are excluded here because their kurtosis DECREASED after cleaning -- the
expected/good direction, not the suspicious pattern being investigated. VP002 had 0 flagged
channels in both RAW and PREPROCESSED, so there's nothing to test.)

For each subject, uses that subject's ACTUAL "PREPROCESSED: high-kurtosis channels" list from
your QC report (hardcoded below, taken directly from your QC output -- not guessed), loads its
RAW and PREPROCESSED (.fif) signals, and for every flagged channel checks whether the extreme
(99.9th percentile) sample amplitudes shrank proportionally with the overall variance (benign) or
disproportionately less (concerning -- residual spikes not fully removed by preprocessing).

Requires each subject's {subject}_eeg_clean_raw.fif to already exist in OUTPUT_PATH/{subject}/
(the file preprocessing_v2.py now saves automatically -- see earlier fix). If a subject only has
the OLD-style output without that .fif, this script will skip it with a clear message rather than
silently falling back to something less exact.

Output: a per-channel verdict, a per-subject summary, and a final overall summary table.
"""

import os
import numpy as np
import scipy.io as sio
from scipy.signal import resample
from scipy.stats import kurtosis
import mne

# ============================================================================
# EDIT THESE TO MATCH YOUR MAIN PREPROCESSING SCRIPT EXACTLY
# ============================================================================
BASE_DATASET_PATH = r"E:\kntu\Thesis\codes\DATA"
OUTPUT_PATH       = r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"
SFREQ_EEG = 200.0
# ============================================================================

# Actual "PREPROCESSED: high-kurtosis channels" lists, copied directly from your QC report
# (the run AFTER the QC script fixes) for every subject whose kurtosis count INCREASED after
# cleaning. Edit/extend this if you check more subjects later.
SUBJECT_FLAGGED_CHANNELS = {
    "VP004": ['Fp1', 'AFF5h', 'AFz', 'F1', 'FC5', 'FC1', 'T7', 'C3', 'Cz', 'CP5', 'CP1', 'P3',
              'Pz', 'POz', 'Fp2', 'AFF6h', 'F2', 'FC2', 'FC6', 'C4', 'T8', 'CP2', 'CP6', 'P4'],
    "VP005": ['Fp1', 'AFF5h', 'F1', 'FC5', 'T7', 'C3', 'CP5', 'Fp2', 'AFF6h', 'FC2', 'FC6',
              'C4', 'T8', 'CP6'],
    "VP006": ['Fp1', 'AFF5h', 'FC5', 'T7', 'C3', 'Cz', 'CP5', 'CP1', 'P7', 'P3', 'Fp2', 'AFF6h',
              'FC6', 'C4', 'T8', 'CP6', 'P4'],
    "VP007": ['FC5', 'C3', 'CP5', 'FC6', 'C4', 'T8', 'CP6'],
    "VP008": ['FC5', 'T7', 'C3', 'Cz', 'FC6', 'C4', 'T8', 'CP6'],
    "VP010": ['Fp1', 'AFF5h', 'AFz', 'F1', 'FC5', 'FC1', 'T7', 'C3', 'CP5', 'CP1', 'P7', 'P3',
              'Pz', 'POz', 'Fp2', 'AFF6h', 'F2', 'FC6', 'C4', 'T8', 'CP2', 'CP6', 'P4', 'P8'],
    "VP011": ['Fp1', 'AFz', 'FC5', 'T7', 'C3', 'Cz', 'CP5', 'CP1', 'Fp2', 'FC6', 'C4', 'CP2'],
}


def load_raw_minimal(subject_id):
    """Minimal RAW loader (no montage needed for this amplitude-only test)."""
    eeg_folder = os.path.join(BASE_DATASET_PATH, f"{subject_id}-EEG")
    cnt_path = os.path.join(eeg_folder, "cnt_nback.mat")
    mat_cnt = sio.loadmat(cnt_path, struct_as_record=False, squeeze_me=True)
    cnt_struct = mat_cnt[[k for k in mat_cnt.keys() if not k.startswith("__")][0]]
    data = cnt_struct.x.T
    sfreq = float(cnt_struct.fs)
    ch_names = [str(ch).strip() for ch in cnt_struct.clab]
    if np.max(np.abs(data)) > 1.0:
        data = data * 1e-6
    if sfreq != SFREQ_EEG:
        n_samples = int(data.shape[1] * SFREQ_EEG / sfreq)
        data = resample(data, n_samples, axis=1)
        sfreq = SFREQ_EEG
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types="eeg")
    return mne.io.RawArray(data, info, verbose=False)


def analyze_kurtosis_increase(raw, clean, ch_name, verbose=True):
    if ch_name not in clean.ch_names:
        if verbose:
            print(f"    [{ch_name}] not present in cleaned data (dropped as bad/interpolated context) -- skipped")
        return None
    raw_data = raw.get_data(picks=[ch_name])[0]
    clean_data = clean.get_data(picks=[ch_name])[0]
    if np.median(np.abs(raw_data)) < 1e-3:
        raw_data = raw_data * 1e6
    if np.median(np.abs(clean_data)) < 1e-3:
        clean_data = clean_data * 1e6

    raw_std, clean_std = raw_data.std(), clean_data.std()
    raw_kurt = kurtosis(raw_data, fisher=True)
    clean_kurt = kurtosis(clean_data, fisher=True)
    raw_p999 = np.percentile(np.abs(raw_data), 99.9)
    clean_p999 = np.percentile(np.abs(clean_data), 99.9)
    variance_shrink_ratio = clean_std / raw_std if raw_std > 0 else np.nan
    spike_shrink_ratio = clean_p999 / raw_p999 if raw_p999 > 0 else np.nan

    if spike_shrink_ratio > variance_shrink_ratio * 1.5:
        verdict = "BENIGN"
    elif spike_shrink_ratio < variance_shrink_ratio * 0.7:
        verdict = "CONCERNING"
    else:
        verdict = "BENIGN"  # proportional shrink -- expected/benign

    if verbose:
        print(f"    [{ch_name}] std {raw_std:.1f}->{clean_std:.1f}uV ({variance_shrink_ratio*100:.0f}%) | "
              f"kurt {raw_kurt:.1f}->{clean_kurt:.1f} | "
              f"p99.9 {raw_p999:.1f}->{clean_p999:.1f}uV ({spike_shrink_ratio*100:.0f}%) -> {verdict}")

    return dict(channel=ch_name, verdict=verdict, raw_std=raw_std, clean_std=clean_std,
                raw_kurt=raw_kurt, clean_kurt=clean_kurt,
                variance_shrink_pct=variance_shrink_ratio * 100, spike_shrink_pct=spike_shrink_ratio * 100)


# ----------------------------------------------------------------------------
# BATCH RUN across all suspicious subjects
# ----------------------------------------------------------------------------
all_results = {}

for subject_id, flagged_channels in SUBJECT_FLAGGED_CHANNELS.items():
    print(f"\n{'='*90}\n{subject_id}  ({len(flagged_channels)} flagged channels)\n{'='*90}")

    clean_fif_path = os.path.join(OUTPUT_PATH, subject_id, f"{subject_id}_eeg_clean_raw.fif")
    if not os.path.exists(clean_fif_path):
        print(f"  SKIPPED: {clean_fif_path} not found. Re-run preprocessing for {subject_id} "
              f"with the .fif-saving fix first (see earlier chat history).")
        continue

    try:
        raw = load_raw_minimal(subject_id)
        clean = mne.io.read_raw_fif(clean_fif_path, preload=True, verbose=False)
    except Exception as e:
        print(f"  SKIPPED: failed to load {subject_id}: {e}")
        continue

    subject_results = []
    for ch in flagged_channels:
        r = analyze_kurtosis_increase(raw, clean, ch)
        if r is not None:
            subject_results.append(r)

    all_results[subject_id] = subject_results
    n_benign = sum(1 for r in subject_results if r["verdict"] == "BENIGN")
    n_concerning = sum(1 for r in subject_results if r["verdict"] == "CONCERNING")
    print(f"  -> {subject_id} summary: {n_benign} benign, {n_concerning} concerning "
          f"out of {len(subject_results)} tested channels")


# ----------------------------------------------------------------------------
# OVERALL SUMMARY
# ----------------------------------------------------------------------------
print(f"\n{'='*90}\nOVERALL SUMMARY (all subjects)\n{'='*90}")
print(f"{'Subject':<10} {'Tested':>8} {'Benign':>8} {'Concerning':>12} {'Concerning channels'}")
total_benign, total_concerning = 0, 0
for subject_id, results in all_results.items():
    n_benign = sum(1 for r in results if r["verdict"] == "BENIGN")
    n_concerning = sum(1 for r in results if r["verdict"] == "CONCERNING")
    concerning_chs = [r["channel"] for r in results if r["verdict"] == "CONCERNING"]
    total_benign += n_benign
    total_concerning += n_concerning
    print(f"{subject_id:<10} {len(results):>8} {n_benign:>8} {n_concerning:>12}   {concerning_chs}")

print(f"\nTOTAL across all tested subjects: {total_benign} benign, {total_concerning} concerning "
      f"({total_benign + total_concerning} channels tested)")

if total_concerning == 0:
    print("\nCONCLUSION: every flagged channel's kurtosis increase is explained by the benign "
          "variance-normalization effect (extreme-sample amplitude shrank proportionally with "
          "overall variance). No evidence of un-removed residual artifacts from this test.")
else:
    print(f"\nCONCLUSION: {total_concerning} channel(s) show a genuine 'concerning' pattern "
          f"(extreme samples did NOT shrink proportionally) -- inspect these subjects/channels "
          f"visually (time_series_snippet.png, ica_components_*.png) before proceeding to modeling.")


VP004  (24 flagged channels)
    [Fp1] std 348.4->41.6uV (12%) | kurt 3.3->5.9 | p99.9 1633.8->230.8uV (14%) -> BENIGN
    [AFF5h] std 406.8->44.6uV (11%) | kurt 2.6->9.5 | p99.9 1815.0->296.5uV (16%) -> BENIGN
    [AFz] std 263.5->16.4uV (6%) | kurt 3.7->7.6 | p99.9 1369.1->95.1uV (7%) -> BENIGN
    [F1] std 261.6->11.4uV (4%) | kurt 2.6->9.6 | p99.9 1242.6->77.4uV (6%) -> BENIGN
    [FC5] std 315.3->32.6uV (10%) | kurt 3.8->9.8 | p99.9 1690.6->218.0uV (13%) -> BENIGN
    [FC1] std 303.5->22.1uV (7%) | kurt 2.7->14.0 | p99.9 1418.1->157.8uV (11%) -> BENIGN
    [T7] std 348.1->23.8uV (7%) | kurt 2.0->12.0 | p99.9 1547.0->175.0uV (11%) -> BENIGN
    [C3] std 319.7->22.9uV (7%) | kurt 4.7->5.6 | p99.9 1812.6->125.6uV (7%) -> BENIGN
    [Cz] std 226.6->34.5uV (15%) | kurt 4.3->6.0 | p99.9 1196.9->195.1uV (16%) -> BENIGN
    [CP5] std 433.7->13.8uV (3%) | kurt 3.4->6.1 | p99.9 2278.7->83.0uV (4%) -> BENIGN
    [CP1] std 283.9->45.6uV (16%) | kurt 4.0->7.1 | p99.9 1427.4->268.4uV (19%) -> 

In [ ]:
#Systematic channel test

In [5]:
"""
SELF-CONTAINED CELL -- Systematic, statistical test for the "repeated low-correlation channel"
hypothesis (e.g. FC6/T8/CP6 in VP001, Fp2/AFF6h in VP004, P8 in VP010).

QUESTION BEING TESTED: are specific channels flagged as "low inter-channel correlation" across
MULTIPLE subjects more often than chance -- suggesting a systematic cause (bad electrode
position, chronic EMG contamination at that scalp location) rather than random per-subject noise?

METHOD (three independent lines of evidence, not just eyeballing a repeated name):

1. FREQUENCY COUNT: for every subject with a saved {subject}_eeg_clean_raw.fif, recompute the
   CAR-adjusted low-correlation flag (identical logic to eeg_preprocessing_qc.py's
   check_channel_correlation, so results are directly comparable to your QC reports) and tally,
   per channel, how many subjects flag it.

2. BINOMIAL SIGNIFICANCE TEST: under the null hypothesis that every channel is equally likely to
   be flagged (i.e. flagging is driven by independent per-subject noise, not a channel-specific
   cause), each channel's flag count should follow Binomial(n_subjects, p_flagged_overall). A
   channel flagged in significantly MORE subjects than this null predicts (one-sided binomial
   test, p<0.05, Bonferroni-corrected across n_channels comparisons) is statistically unlikely to
   be coincidental.

3. MUSCLE-CONTAMINATION CROSS-CHECK: for every channel, compute its mean relative power in the
   45-100 Hz ("muscle_hf") band across all subjects. If channels with significantly repeated
   low-correlation flags ALSO have significantly higher muscle-band power than other channels
   (independent-samples t-test), this is direct spectral evidence supporting the EMG/muscle-
   proximity hypothesis (T7/T8, FC5/FC6, CP5/CP6 sit over temporalis/masseter -- chewing/jaw
   muscles -- and Fp1/Fp2 sit over frontalis/corrugator -- forehead/brow muscles), rather than a
   generic "bad electrode" explanation.

Run this after preprocessing has produced {subject}_eeg_clean_raw.fif for the subjects you want
to include (ideally all subjects that have it).
"""

import os
import numpy as np
from scipy import signal
from scipy.stats import binomtest, ttest_ind
import mne

# ============================================================================
BASE_DATASET_PATH = r"E:\kntu\Thesis\codes\DATA"
OUTPUT_PATH       = r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"
SUBJECTS_TO_CHECK = [f"VP{n:03d}" for n in range(1, 27) if n != 4]  # excludes VP004 (already excluded from training)
CORR_MARGIN_BELOW_CAR_BASELINE = 0.15  # must match eeg_preprocessing_qc.py's value
ALPHA = 0.05  # significance level before Bonferroni correction
MUSCLE_BAND = (45.0, 100.0)
# ============================================================================


def get_low_corr_channels(clean_raw):
    """Identical logic to eeg_preprocessing_qc.py's check_channel_correlation (CAR-adjusted)."""
    data = clean_raw.get_data(picks="eeg")
    if np.median(np.abs(data)) < 1e-3:
        data = data * 1e6
    n_ch = data.shape[0]
    corr = np.corrcoef(data)
    np.fill_diagonal(corr, np.nan)
    mean_corr_per_ch = np.nanmean(corr, axis=1)

    is_car = bool(clean_raw.info.get("custom_ref_applied", False))
    expected_baseline = -1.0 / max(n_ch - 1, 1)
    effective_threshold = (expected_baseline - CORR_MARGIN_BELOW_CAR_BASELINE) if is_car else 0.4

    ch_names = clean_raw.ch_names[:n_ch]
    return [ch_names[i] for i, c in enumerate(mean_corr_per_ch) if c < effective_threshold]


def get_muscle_band_power(clean_raw):
    """Per-channel relative power in the 45-100Hz muscle/EMG band."""
    data = clean_raw.get_data(picks="eeg")
    if np.median(np.abs(data)) < 1e-3:
        data = data * 1e6
    sfreq = clean_raw.info["sfreq"]
    freqs, psd = signal.welch(data, fs=sfreq, nperseg=min(int(4 * sfreq), data.shape[1]), axis=1)
    mask = (freqs >= MUSCLE_BAND[0]) & (freqs <= min(MUSCLE_BAND[1], sfreq / 2 - 1))
    band_power = psd[:, mask].mean(axis=1)
    total_power = psd.mean(axis=1)
    relative_power = band_power / (total_power + 1e-20)  # relative, so it's comparable across subjects/scales
    ch_names = clean_raw.ch_names[:data.shape[0]]
    return dict(zip(ch_names, relative_power))


# ----------------------------------------------------------------------------
# STEP 1: frequency count across all available subjects
# ----------------------------------------------------------------------------
channel_flag_counts = {}
channel_muscle_power = {}  # channel -> list of relative muscle-band power values, one per subject
n_subjects_checked = 0

for subject_id in SUBJECTS_TO_CHECK:
    clean_fif_path = os.path.join(OUTPUT_PATH, subject_id, f"{subject_id}_eeg_clean_raw.fif")
    if not os.path.exists(clean_fif_path):
        continue
    try:
        clean = mne.io.read_raw_fif(clean_fif_path, preload=True, verbose=False)
    except Exception as e:
        print(f"  SKIPPED {subject_id}: {e}")
        continue

    n_subjects_checked += 1
    low_corr = get_low_corr_channels(clean)
    for ch in low_corr:
        channel_flag_counts[ch] = channel_flag_counts.get(ch, 0) + 1

    muscle_power = get_muscle_band_power(clean)
    for ch, p in muscle_power.items():
        channel_muscle_power.setdefault(ch, []).append(p)

print(f"Checked {n_subjects_checked} subjects with available {{subject}}_eeg_clean_raw.fif\n")

if n_subjects_checked == 0:
    raise RuntimeError("No subjects with saved _eeg_clean_raw.fif were found -- nothing to test.")

# overall base flagging rate (for the binomial null model)
total_flags = sum(channel_flag_counts.values())
all_channels = sorted(channel_muscle_power.keys())
n_channels = len(all_channels)
p_null = total_flags / (n_subjects_checked * n_channels)  # expected per-channel-per-subject flag probability under uniform null

print(f"Overall base flagging rate (null model): p={p_null:.4f} "
      f"({total_flags} total flags / {n_subjects_checked} subjects / {n_channels} channels)\n")

# ----------------------------------------------------------------------------
# STEP 2: binomial significance test per channel, Bonferroni-corrected
# ----------------------------------------------------------------------------
alpha_corrected = ALPHA / n_channels

print(f"{'Channel':<10} {'Flagged in':>12} {'p-value':>12} {'Bonf. sig?':>12} {'Mean muscle_hf rel.power':>26}")
print("-" * 80)

significant_channels = []
results_table = []

for ch in all_channels:
    count = channel_flag_counts.get(ch, 0)
    if n_subjects_checked > 0:
        test = binomtest(count, n_subjects_checked, p_null, alternative="greater")
        pval = test.pvalue
    else:
        pval = np.nan
    is_sig = pval < alpha_corrected
    mean_muscle_power = np.mean(channel_muscle_power[ch]) if ch in channel_muscle_power else np.nan

    results_table.append(dict(channel=ch, n_flagged=count, n_subjects=n_subjects_checked,
                               pvalue=pval, significant=is_sig, mean_muscle_power=mean_muscle_power))
    if is_sig:
        significant_channels.append(ch)

    if count > 0:  # only print channels that were flagged at least once (keeps output readable)
        print(f"{ch:<10} {count:>5}/{n_subjects_checked:<6} {pval:>12.4g} {'YES' if is_sig else 'no':>12} "
              f"{mean_muscle_power:>26.4f}")

print(f"\nBonferroni-corrected significance threshold: p < {alpha_corrected:.4g} "
      f"(alpha={ALPHA} / {n_channels} channels)")
print(f"\nStatistically significant channels (flagged far more than chance): {significant_channels}")

# ----------------------------------------------------------------------------
# STEP 3: muscle-band cross-check -- do significant channels have higher muscle_hf power?
# ----------------------------------------------------------------------------
if significant_channels:
    sig_powers = [np.mean(channel_muscle_power[ch]) for ch in significant_channels if ch in channel_muscle_power]
    other_channels = [ch for ch in all_channels if ch not in significant_channels]
    other_powers = [np.mean(channel_muscle_power[ch]) for ch in other_channels if ch in channel_muscle_power]

    t_stat, t_pval = ttest_ind(sig_powers, other_powers, equal_var=False)
    print(f"\nMUSCLE-BAND CROSS-CHECK:")
    print(f"  Mean muscle_hf relative power -- significant channels: {np.mean(sig_powers):.4f} "
          f"(n={len(sig_powers)})")
    print(f"  Mean muscle_hf relative power -- other channels:       {np.mean(other_powers):.4f} "
          f"(n={len(other_powers)})")
    print(f"  Welch's t-test: t={t_stat:.3f}, p={t_pval:.4g}")
    if t_pval < 0.05 and np.mean(sig_powers) > np.mean(other_powers):
        print("  -> SUPPORTS the EMG/muscle-proximity hypothesis: repeatedly-flagged channels have "
              "significantly higher muscle-band power than other channels.")
    elif t_pval < 0.05:
        print("  -> Significant difference, but in the OPPOSITE direction -- muscle contamination is "
              "NOT the likely explanation for these channels; investigate electrode/impedance causes instead.")
    else:
        print("  -> No significant muscle-band difference -- the repeated flagging is not clearly "
              "explained by EMG contamination; consider electrode placement/impedance as the cause instead.")
else:
    print("\nNo channel reached Bonferroni-corrected significance -- the repeated low-correlation "
          "pattern observed in VP001/VP004/VP010 does not hold up as a systematic, dataset-wide "
          "effect once tested rigorously across all subjects. Likely per-subject coincidence/noise.")

# ----------------------------------------------------------------------------
# FULL RANKED TABLE (all channels, sorted by flag frequency)
# ----------------------------------------------------------------------------
print(f"\n{'='*80}\nFULL RANKING (all channels, sorted by how often flagged)\n{'='*80}")
results_table.sort(key=lambda r: -r["n_flagged"])
for r in results_table:
    if r["n_flagged"] > 0:
        print(f"{r['channel']:<10} flagged {r['n_flagged']}/{r['n_subjects']} subjects "
              f"({r['n_flagged']/r['n_subjects']*100:.1f}%) | p={r['pvalue']:.4g} | "
              f"{'SIGNIFICANT' if r['significant'] else 'not significant'}")

Checked 25 subjects with available {subject}_eeg_clean_raw.fif

Overall base flagging rate (null model): p=0.0200 (14 total flags / 25 subjects / 28 channels)

Channel      Flagged in      p-value   Bonf. sig?   Mean muscle_hf rel.power
--------------------------------------------------------------------------------
AFF5h          1/25           0.3965           no                     0.0026
CP6            1/25           0.3965           no                     0.0042
FC5            1/25           0.3965           no                     0.0046
FC6            1/25           0.3965           no                     0.0036
Fp1            4/25         0.001446          YES                     0.0023
Fp2            3/25          0.01324           no                     0.0024
P8             1/25           0.3965           no                     0.0022
T7             1/25           0.3965           no                     0.0050
T8             1/25           0.3965           no                 

In [ ]:
#Delta band ocular crosscheck

In [6]:
"""
SELF-CONTAINED CELL -- Delta-band (0.5-4Hz) cross-check for the Fp1/Fp2 ocular-artifact
hypothesis, correcting the previous (wrong-band) muscle_hf cross-check.

WHY THIS TEST: ocular artifacts (eye blinks, slow eye movements) are dominated by LOW-frequency
content (<4Hz), not the 45-100Hz muscle band used previously. If the subjects whose Fp1/Fp2
channels were flagged as "low inter-channel correlation" have GENUINELY elevated residual ocular
contamination, their Fp1/Fp2 delta-band power should be elevated relative to (a) other channels
in the SAME subject, and (b) Fp1/Fp2 in subjects that were NOT flagged.

METHOD (two independent comparisons per channel):
  A) WITHIN-SUBJECT: for each flagged subject, is Fp1's (or Fp2's) delta-band relative power an
     outlier compared to that same subject's other channels? (paired-style, controls for
     subject-level baseline differences)
  B) BETWEEN-SUBJECT: is Fp1's (or Fp2's) delta-band relative power, across the FLAGGED subjects,
     significantly higher than Fp1's delta-band power across the NON-flagged subjects?
     (independent-samples t-test, tests whether this is a subject-specific vs a general effect)

Requires {subject}_eeg_clean_raw.fif for each subject checked (same requirement as the previous
correlation test).
"""

import os
import numpy as np
from scipy import signal
from scipy.stats import ttest_ind
import mne

# ============================================================================
BASE_DATASET_PATH = r"E:\kntu\Thesis\codes\DATA"
OUTPUT_PATH       = r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"
SUBJECTS_TO_CHECK = [f"VP{n:03d}" for n in range(1, 27) if n != 4]  # VP004 already excluded
CORR_MARGIN_BELOW_CAR_BASELINE = 0.15  # must match eeg_preprocessing_qc.py / systematic_channel_test.py
TARGET_CHANNELS = ["Fp1", "Fp2"]        # the channels flagged as statistically repeated
DELTA_BAND = (0.5, 4.0)
# ============================================================================


def get_low_corr_channels(clean_raw):
    data = clean_raw.get_data(picks="eeg")
    if np.median(np.abs(data)) < 1e-3:
        data = data * 1e6
    n_ch = data.shape[0]
    corr = np.corrcoef(data)
    np.fill_diagonal(corr, np.nan)
    mean_corr_per_ch = np.nanmean(corr, axis=1)
    is_car = bool(clean_raw.info.get("custom_ref_applied", False))
    expected_baseline = -1.0 / max(n_ch - 1, 1)
    effective_threshold = (expected_baseline - CORR_MARGIN_BELOW_CAR_BASELINE) if is_car else 0.4
    ch_names = clean_raw.ch_names[:n_ch]
    return [ch_names[i] for i, c in enumerate(mean_corr_per_ch) if c < effective_threshold]


def get_relative_band_power(clean_raw, band):
    data = clean_raw.get_data(picks="eeg")
    if np.median(np.abs(data)) < 1e-3:
        data = data * 1e6
    sfreq = clean_raw.info["sfreq"]
    freqs, psd = signal.welch(data, fs=sfreq, nperseg=min(int(4 * sfreq), data.shape[1]), axis=1)
    mask = (freqs >= band[0]) & (freqs <= min(band[1], sfreq / 2 - 1))
    band_power = psd[:, mask].mean(axis=1)
    total_power = psd.mean(axis=1)
    relative_power = band_power / (total_power + 1e-20)
    ch_names = clean_raw.ch_names[:data.shape[0]]
    return dict(zip(ch_names, relative_power))


# ----------------------------------------------------------------------------
# Pass 1: figure out exactly which subjects flagged each target channel, and collect delta power
# ----------------------------------------------------------------------------
flagged_subjects = {ch: [] for ch in TARGET_CHANNELS}      # channel -> list of subject_ids that flagged it
target_delta_power = {ch: {} for ch in TARGET_CHANNELS}     # channel -> {subject_id: delta relative power}
within_subject_other_delta = {ch: {} for ch in TARGET_CHANNELS}  # channel -> {subject_id: mean delta power of OTHER channels}

n_checked = 0
for subject_id in SUBJECTS_TO_CHECK:
    clean_fif_path = os.path.join(OUTPUT_PATH, subject_id, f"{subject_id}_eeg_clean_raw.fif")
    if not os.path.exists(clean_fif_path):
        continue
    try:
        clean = mne.io.read_raw_fif(clean_fif_path, preload=True, verbose=False)
    except Exception as e:
        print(f"  SKIPPED {subject_id}: {e}")
        continue
    n_checked += 1

    low_corr = get_low_corr_channels(clean)
    delta_power = get_relative_band_power(clean, DELTA_BAND)

    for ch in TARGET_CHANNELS:
        if ch in low_corr:
            flagged_subjects[ch].append(subject_id)
        if ch in delta_power:
            target_delta_power[ch][subject_id] = delta_power[ch]
            other_vals = [v for c, v in delta_power.items() if c != ch]
            within_subject_other_delta[ch][subject_id] = np.mean(other_vals)

print(f"Checked {n_checked} subjects.\n")

# ----------------------------------------------------------------------------
# Test A (within-subject) and Test B (between-subject), per target channel
# ----------------------------------------------------------------------------
for ch in TARGET_CHANNELS:
    flagged = flagged_subjects[ch]
    print(f"\n{'='*90}\n{ch}: flagged as low-correlation in {len(flagged)} subject(s): {flagged}\n{'='*90}")

    if len(flagged) == 0:
        print("  No subjects flagged this channel -- nothing to test.")
        continue

    # --- Test A: within-subject, for each flagged subject, is this channel's delta power an
    #     outlier vs. that subject's own other channels? ---
    print("\n  TEST A (within-subject): channel delta power vs. that subject's other channels")
    for sid in flagged:
        ch_val = target_delta_power[ch].get(sid)
        other_val = within_subject_other_delta[ch].get(sid)
        if ch_val is None or other_val is None:
            continue
        ratio = ch_val / other_val if other_val > 0 else np.nan
        flag = "ELEVATED" if ratio > 1.5 else ("REDUCED" if ratio < 0.67 else "comparable")
        print(f"    {sid}: {ch} delta rel.power={ch_val:.4f} vs subject's other-channel mean={other_val:.4f} "
              f"(ratio={ratio:.2f}x) -> {flag}")

    # --- Test B: between-subject, is this channel's delta power higher in flagged subjects than
    #     in non-flagged subjects? ---
    non_flagged = [sid for sid in target_delta_power[ch] if sid not in flagged]
    flagged_vals = [target_delta_power[ch][sid] for sid in flagged if sid in target_delta_power[ch]]
    non_flagged_vals = [target_delta_power[ch][sid] for sid in non_flagged]

    print(f"\n  TEST B (between-subject): {ch} delta power, flagged vs non-flagged subjects")
    print(f"    Flagged subjects (n={len(flagged_vals)}): mean={np.mean(flagged_vals):.4f}, "
          f"values={[round(v,4) for v in flagged_vals]}")
    print(f"    Non-flagged subjects (n={len(non_flagged_vals)}): mean={np.mean(non_flagged_vals):.4f}")

    if len(flagged_vals) >= 2 and len(non_flagged_vals) >= 2:
        t_stat, p_val = ttest_ind(flagged_vals, non_flagged_vals, equal_var=False)
        print(f"    Welch's t-test: t={t_stat:.3f}, p={p_val:.4g}")
        if p_val < 0.05 and np.mean(flagged_vals) > np.mean(non_flagged_vals):
            print(f"    -> SUPPORTS the ocular-artifact hypothesis: {ch} delta power is significantly "
                  f"elevated in the flagged subjects.")
        elif p_val < 0.05:
            print(f"    -> Significant, but LOWER delta power in flagged subjects -- does NOT support "
                  f"the ocular hypothesis; investigate another cause.")
        else:
            print(f"    -> Not statistically significant at group level (n={len(flagged_vals)} is small, "
                  f"so this test has limited power) -- rely more on Test A (within-subject) above.")
    else:
        print(f"    -> Too few flagged subjects (n={len(flagged_vals)}) for a reliable group t-test; "
              f"interpret Test A (within-subject) results instead.")

print(f"\n{'='*90}\nSUMMARY / INTERPRETATION GUIDE\n{'='*90}")
print("""
- If Test A shows 'ELEVATED' (ratio > 1.5x) for most/all flagged subjects, AND Test B is
  significant (or trends strongly the same direction despite small n), this is solid evidence
  the low-correlation flag at Fp1/Fp2 reflects genuine residual ocular contamination, not a
  coincidence or generic electrode issue.
- If ratios are 'comparable' or 'REDUCED', the ocular hypothesis is not supported either, and the
  cause is more likely electrode impedance/placement specific to those subjects -- worth a manual
  visual check of time_series_snippet.png for exactly these subject/channel pairs.
""")

Checked 25 subjects.


Fp1: flagged as low-correlation in 4 subject(s): ['VP015', 'VP018', 'VP019', 'VP020']

  TEST A (within-subject): channel delta power vs. that subject's other channels
    VP015: Fp1 delta rel.power=18.6276 vs subject's other-channel mean=12.0004 (ratio=1.55x) -> ELEVATED
    VP018: Fp1 delta rel.power=14.1983 vs subject's other-channel mean=7.0576 (ratio=2.01x) -> ELEVATED
    VP019: Fp1 delta rel.power=15.4907 vs subject's other-channel mean=12.1959 (ratio=1.27x) -> comparable
    VP020: Fp1 delta rel.power=17.2715 vs subject's other-channel mean=9.6643 (ratio=1.79x) -> ELEVATED

  TEST B (between-subject): Fp1 delta power, flagged vs non-flagged subjects
    Flagged subjects (n=4): mean=16.3970, values=[np.float64(18.6276), np.float64(14.1983), np.float64(15.4907), np.float64(17.2715)]
    Non-flagged subjects (n=21): mean=11.9737
    Welch's t-test: t=3.508, p=0.008191
    -> SUPPORTS the ocular-artifact hypothesis: Fp1 delta power is significantly elevated i

In [ ]:
#Ica eyeblink probability deepdive · PY

In [7]:
"""
SELF-CONTAINED CELL -- for the 4 subjects flagged with elevated Fp1/Fp2 delta-band power
(VP015, VP018, VP019, VP020), refits ICA with the SAME parameters as your main pipeline and
prints the FULL ICLabel probability breakdown for every component -- not just which ones crossed
the 0.30 threshold, but the exact eye blink / eye movement probability for ALL components, so we
can see whether a genuine ocular component exists just below the cutoff (rescuable by lowering
the threshold) or whether no component captures this pattern at all (meaning Fp1/Fp2
interpolation is the correct fix instead).

Cross-validated against ica.find_bads_eog() (VEOG/HEOG correlation), which is an INDEPENDENT
detection method from ICLabel -- if both methods agree a component is borderline-ocular, that's
much stronger evidence than either alone.

Requires the raw dataset files (BASE_DATASET_PATH) -- reprocesses these 4 subjects' ICA from
scratch (does NOT touch/overwrite any saved preprocessing output files).
"""

import os
import numpy as np
import scipy.io as sio
from scipy.signal import resample
import mne
import mne_icalabel

# ============================================================================
BASE_DATASET_PATH = r"E:\kntu\Thesis\codes\DATA"
SFREQ_EEG = 200.0
TARGET_SUBJECTS = ["VP015", "VP018", "VP019", "VP020"]
CURRENT_PROB_THRESHOLD = 0.30   # your pipeline's current ICLabel exclusion threshold
BORDERLINE_LOW = 0.10           # report components with eye-blink/movement prob in this range
                                 # as "borderline" (below current threshold but not negligible)
# ============================================================================

ARTIFACT_CLASSES = ['brain', 'muscle artifact', 'eye blink', 'heart beat',
                    'line noise', 'channel noise', 'other']
EYE_RELATED_LABELS = {'eye blink', 'eye movement'}


def load_eeg_data(eeg_folder):
    cnt_path = os.path.join(eeg_folder, 'cnt_nback.mat')
    mrk_path = os.path.join(eeg_folder, 'mrk_nback.mat')
    mat_cnt = sio.loadmat(cnt_path, struct_as_record=False, squeeze_me=True)
    cnt_struct = mat_cnt[[k for k in mat_cnt.keys() if not k.startswith('__')][0]]
    data = cnt_struct.x.T
    sfreq = float(cnt_struct.fs)
    ch_names = [str(ch).strip() for ch in cnt_struct.clab]
    if np.max(np.abs(data)) > 1.0:
        data = data * 1e-6
    if sfreq != SFREQ_EEG:
        n_samples = int(data.shape[1] * SFREQ_EEG / sfreq)
        data = resample(data, n_samples, axis=1)
        sfreq = SFREQ_EEG
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')
    raw = mne.io.RawArray(data, info, verbose=False)
    eog_channels = [ch for ch in ['VEOG', 'HEOG'] if ch in ch_names]
    if eog_channels:
        raw.set_channel_types({ch: 'eog' for ch in eog_channels})
    raw.set_montage('standard_1005', match_case=False, on_missing='ignore')

    mat_mrk = sio.loadmat(mrk_path, struct_as_record=False, squeeze_me=True)
    mrk_struct = mat_mrk[[k for k in mat_mrk.keys() if not k.startswith('__')][0]]
    if hasattr(mrk_struct, 'time'):
        event_positions = np.round(mrk_struct.time * sfreq / 1000).astype(int)
    else:
        event_positions = np.array(mrk_struct.pos, dtype=int) - 1
    events = np.zeros((len(event_positions), 3), dtype=int)
    events[:, 0] = event_positions
    return raw, events


def fit_ica_and_report(subject_id):
    print(f"\n{'='*95}\n{subject_id}\n{'='*95}")
    eeg_folder = os.path.join(BASE_DATASET_PATH, f"{subject_id}-EEG")
    raw, events = load_eeg_data(eeg_folder)

    # same preprocessing as your main pipeline, up to (not including) component exclusion
    raw_filtered = raw.copy().notch_filter(freqs=50.0, picks="eeg", verbose=False)
    raw_filtered.filter(l_freq=0.5, h_freq=40.0, method="fir", verbose=False)

    raw_temp = raw_filtered.copy().filter(l_freq=10.0, h_freq=40.0, method="fir", verbose=False)
    eeg_picks = mne.pick_types(raw_temp.info, eeg=True, eog=False)
    eeg_names = [raw_temp.ch_names[idx] for idx in eeg_picks]
    temp_data = raw_temp.get_data(picks=eeg_picks)
    stds = np.std(temp_data, axis=1)
    median_std = np.median(stds)
    mad = np.median(np.abs(stds - median_std)) if np.median(np.abs(stds - median_std)) > 0 else 1e-6
    corr_matrix = np.corrcoef(temp_data)
    mean_corrs = np.mean(corr_matrix, axis=1)
    bad_channels = []
    for i, ch_name in enumerate(eeg_names):
        if ch_name.upper() in ["VEOG", "HEOG", "TP9"]:
            continue
        if stds[i] > (median_std + 4.5 * mad) or stds[i] < (median_std - 4.5 * mad) or mean_corrs[i] < 0.30:
            bad_channels.append(ch_name)
    if bad_channels:
        raw_filtered.info["bads"] = bad_channels
    if raw_filtered.info["bads"]:
        raw_filtered.interpolate_bads(reset_bads=True, verbose=False)
    raw_filtered.set_eeg_reference("average", projection=False, verbose=False)

    picks_for_ica = mne.pick_types(raw_filtered.info, eeg=True, eog=False, exclude="bads")
    ica = mne.preprocessing.ICA(n_components=0.99, method="picard",
                                 fit_params=dict(ortho=False, extended=True),
                                 random_state=42, verbose=False)
    ica.fit(raw_filtered, picks=picks_for_ica)

    ic_labels = mne_icalabel.label_components(raw_filtered, ica, method="iclabel")
    proba_matrix = ica.labels_scores_  # (n_components, 7)

    # independent cross-check: VEOG/HEOG correlation (different detection method than ICLabel)
    eog_indices_veog, eog_scores_veog = [], None
    eog_indices_heog, eog_scores_heog = [], None
    try:
        if "VEOG" in raw.ch_names:
            eog_indices_veog, eog_scores_veog = ica.find_bads_eog(raw, ch_name="VEOG", threshold=2.0, verbose=False)
        if "HEOG" in raw.ch_names:
            eog_indices_heog, eog_scores_heog = ica.find_bads_eog(raw, ch_name="HEOG", threshold=2.0, verbose=False)
    except Exception as e:
        print(f"  find_bads_eog failed: {e}")

    eye_col_indices = [ARTIFACT_CLASSES.index(lbl) for lbl in EYE_RELATED_LABELS if lbl in ARTIFACT_CLASSES]

    print(f"{'IC#':<5} {'ICLabel argmax':<18} {'eye-related prob':>16} {'VEOG r':>10} {'HEOG r':>10} {'Verdict':<28}")
    print("-" * 95)

    borderline_found = []
    for comp_idx in range(proba_matrix.shape[0]):
        argmax_label = ic_labels["labels"][comp_idx]
        eye_prob = max(proba_matrix[comp_idx, c] for c in eye_col_indices)
        veog_r = eog_scores_veog[comp_idx] if eog_scores_veog is not None and comp_idx < len(eog_scores_veog) else np.nan
        heog_r = eog_scores_heog[comp_idx] if eog_scores_heog is not None and comp_idx < len(eog_scores_heog) else np.nan

        already_excluded = eye_prob >= CURRENT_PROB_THRESHOLD or comp_idx in eog_indices_veog or comp_idx in eog_indices_heog
        if already_excluded:
            verdict = "already excluded"
        elif eye_prob >= BORDERLINE_LOW:
            verdict = "BORDERLINE -- inspect"
            borderline_found.append((comp_idx, eye_prob, veog_r, heog_r))
        else:
            verdict = "clearly not ocular"

        marker = " <--" if verdict == "BORDERLINE -- inspect" else ""
        print(f"{comp_idx:<5} {argmax_label:<18} {eye_prob:>16.3f} {veog_r:>10.3f} {heog_r:>10.3f} {verdict:<28}{marker}")

    print(f"\n  Summary for {subject_id}:")
    if borderline_found:
        print(f"    {len(borderline_found)} borderline ocular component(s) found (prob in "
              f"[{BORDERLINE_LOW}, {CURRENT_PROB_THRESHOLD})):")
        for comp_idx, eye_prob, veog_r, heog_r in borderline_found:
            corroborated = (not np.isnan(veog_r) and abs(veog_r) > 0.3) or (not np.isnan(heog_r) and abs(heog_r) > 0.3)
            print(f"      IC{comp_idx}: eye-prob={eye_prob:.3f}, VEOG r={veog_r:.3f}, HEOG r={heog_r:.3f} "
                  f"-> {'CORROBORATED by EOG correlation (strong case for lowering threshold)' if corroborated else 'not corroborated by EOG correlation (weaker case)'}")
    else:
        print(f"    No borderline ocular components found -- no ICA component captures this pattern "
              f"even loosely. This means lowering the ICLabel threshold would NOT help; "
              f"Fp1/Fp2 interpolation is the more appropriate fix for {subject_id}.")

    return borderline_found


# ----------------------------------------------------------------------------
# RUN for all 4 target subjects
# ----------------------------------------------------------------------------
all_borderline = {}
for sid in TARGET_SUBJECTS:
    try:
        all_borderline[sid] = fit_ica_and_report(sid)
    except Exception as e:
        print(f"\n{sid}: FAILED -- {e}")
        all_borderline[sid] = None

print(f"\n\n{'='*95}\nOVERALL RECOMMENDATION\n{'='*95}")
subjects_with_corroborated_borderline = []
subjects_needing_interpolation = []
for sid, borderline in all_borderline.items():
    if borderline is None:
        continue
    if len(borderline) == 0:
        subjects_needing_interpolation.append(sid)
    else:
        subjects_with_corroborated_borderline.append(sid)

print(f"Subjects with a borderline ocular IC (lowering threshold may help): {subjects_with_corroborated_borderline}")
print(f"Subjects with NO borderline ocular IC (interpolate Fp1/Fp2 instead): {subjects_needing_interpolation}")


VP015
Fitting ICA to data using 28 channels (please be patient, this may take a while)
Selecting by explained variance: 5 components
Fitting ICA took 8.1s.
IC#   ICLabel argmax     eye-related prob     VEOG r     HEOG r Verdict                     
-----------------------------------------------------------------------------------------------
0     eye blink                     0.940      0.991      0.014 already excluded            
1     other                         0.136      0.006     -0.042 BORDERLINE -- inspect        <--
2     eye blink                     0.995     -0.036     -0.803 already excluded            
3     brain                         0.000     -0.053     -0.018 clearly not ocular          
4     brain                         0.000      0.100      0.011 clearly not ocular          

  Summary for VP015:
    1 borderline ocular component(s) found (prob in [0.1, 0.3)):
      IC1: eye-prob=0.136, VEOG r=0.006, HEOG r=-0.042 -> not corroborated by EOG correlation (wea

In [ ]:
#check2 after reprocess
#Delta band ocular crosscheck

In [8]:
"""
SELF-CONTAINED CELL -- Delta-band (0.5-4Hz) cross-check for the Fp1/Fp2 ocular-artifact
hypothesis, correcting the previous (wrong-band) muscle_hf cross-check.

WHY THIS TEST: ocular artifacts (eye blinks, slow eye movements) are dominated by LOW-frequency
content (<4Hz), not the 45-100Hz muscle band used previously. If the subjects whose Fp1/Fp2
channels were flagged as "low inter-channel correlation" have GENUINELY elevated residual ocular
contamination, their Fp1/Fp2 delta-band power should be elevated relative to (a) other channels
in the SAME subject, and (b) Fp1/Fp2 in subjects that were NOT flagged.

METHOD (two independent comparisons per channel):
  A) WITHIN-SUBJECT: for each flagged subject, is Fp1's (or Fp2's) delta-band relative power an
     outlier compared to that same subject's other channels? (paired-style, controls for
     subject-level baseline differences)
  B) BETWEEN-SUBJECT: is Fp1's (or Fp2's) delta-band relative power, across the FLAGGED subjects,
     significantly higher than Fp1's delta-band power across the NON-flagged subjects?
     (independent-samples t-test, tests whether this is a subject-specific vs a general effect)

Requires {subject}_eeg_clean_raw.fif for each subject checked (same requirement as the previous
correlation test).
"""

import os
import numpy as np
from scipy import signal
from scipy.stats import ttest_ind
import mne

# ============================================================================
BASE_DATASET_PATH = r"E:\kntu\Thesis\codes\DATA"
OUTPUT_PATH       = r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"
SUBJECTS_TO_CHECK = [f"VP{n:03d}" for n in range(1, 27) if n != 4]  # VP004 already excluded
CORR_MARGIN_BELOW_CAR_BASELINE = 0.15  # must match eeg_preprocessing_qc.py / systematic_channel_test.py
TARGET_CHANNELS = ["Fp1", "Fp2"]        # the channels flagged as statistically repeated
DELTA_BAND = (0.5, 4.0)
# ============================================================================


def get_low_corr_channels(clean_raw):
    data = clean_raw.get_data(picks="eeg")
    if np.median(np.abs(data)) < 1e-3:
        data = data * 1e6
    n_ch = data.shape[0]
    corr = np.corrcoef(data)
    np.fill_diagonal(corr, np.nan)
    mean_corr_per_ch = np.nanmean(corr, axis=1)
    is_car = bool(clean_raw.info.get("custom_ref_applied", False))
    expected_baseline = -1.0 / max(n_ch - 1, 1)
    effective_threshold = (expected_baseline - CORR_MARGIN_BELOW_CAR_BASELINE) if is_car else 0.4
    ch_names = clean_raw.ch_names[:n_ch]
    return [ch_names[i] for i, c in enumerate(mean_corr_per_ch) if c < effective_threshold]


def get_relative_band_power(clean_raw, band):
    data = clean_raw.get_data(picks="eeg")
    if np.median(np.abs(data)) < 1e-3:
        data = data * 1e6
    sfreq = clean_raw.info["sfreq"]
    freqs, psd = signal.welch(data, fs=sfreq, nperseg=min(int(4 * sfreq), data.shape[1]), axis=1)
    mask = (freqs >= band[0]) & (freqs <= min(band[1], sfreq / 2 - 1))
    band_power = psd[:, mask].mean(axis=1)
    total_power = psd.mean(axis=1)
    relative_power = band_power / (total_power + 1e-20)
    ch_names = clean_raw.ch_names[:data.shape[0]]
    return dict(zip(ch_names, relative_power))


# ----------------------------------------------------------------------------
# Pass 1: figure out exactly which subjects flagged each target channel, and collect delta power
# ----------------------------------------------------------------------------
flagged_subjects = {ch: [] for ch in TARGET_CHANNELS}      # channel -> list of subject_ids that flagged it
target_delta_power = {ch: {} for ch in TARGET_CHANNELS}     # channel -> {subject_id: delta relative power}
within_subject_other_delta = {ch: {} for ch in TARGET_CHANNELS}  # channel -> {subject_id: mean delta power of OTHER channels}

n_checked = 0
for subject_id in SUBJECTS_TO_CHECK:
    clean_fif_path = os.path.join(OUTPUT_PATH, subject_id, f"{subject_id}_eeg_clean_raw.fif")
    if not os.path.exists(clean_fif_path):
        continue
    try:
        clean = mne.io.read_raw_fif(clean_fif_path, preload=True, verbose=False)
    except Exception as e:
        print(f"  SKIPPED {subject_id}: {e}")
        continue
    n_checked += 1

    low_corr = get_low_corr_channels(clean)
    delta_power = get_relative_band_power(clean, DELTA_BAND)

    for ch in TARGET_CHANNELS:
        if ch in low_corr:
            flagged_subjects[ch].append(subject_id)
        if ch in delta_power:
            target_delta_power[ch][subject_id] = delta_power[ch]
            other_vals = [v for c, v in delta_power.items() if c != ch]
            within_subject_other_delta[ch][subject_id] = np.mean(other_vals)

print(f"Checked {n_checked} subjects.\n")

# ----------------------------------------------------------------------------
# Test A (within-subject) and Test B (between-subject), per target channel
# ----------------------------------------------------------------------------
for ch in TARGET_CHANNELS:
    flagged = flagged_subjects[ch]
    print(f"\n{'='*90}\n{ch}: flagged as low-correlation in {len(flagged)} subject(s): {flagged}\n{'='*90}")

    if len(flagged) == 0:
        print("  No subjects flagged this channel -- nothing to test.")
        continue

    # --- Test A: within-subject, for each flagged subject, is this channel's delta power an
    #     outlier vs. that subject's own other channels? ---
    print("\n  TEST A (within-subject): channel delta power vs. that subject's other channels")
    for sid in flagged:
        ch_val = target_delta_power[ch].get(sid)
        other_val = within_subject_other_delta[ch].get(sid)
        if ch_val is None or other_val is None:
            continue
        ratio = ch_val / other_val if other_val > 0 else np.nan
        flag = "ELEVATED" if ratio > 1.5 else ("REDUCED" if ratio < 0.67 else "comparable")
        print(f"    {sid}: {ch} delta rel.power={ch_val:.4f} vs subject's other-channel mean={other_val:.4f} "
              f"(ratio={ratio:.2f}x) -> {flag}")

    # --- Test B: between-subject, is this channel's delta power higher in flagged subjects than
    #     in non-flagged subjects? ---
    non_flagged = [sid for sid in target_delta_power[ch] if sid not in flagged]
    flagged_vals = [target_delta_power[ch][sid] for sid in flagged if sid in target_delta_power[ch]]
    non_flagged_vals = [target_delta_power[ch][sid] for sid in non_flagged]

    print(f"\n  TEST B (between-subject): {ch} delta power, flagged vs non-flagged subjects")
    print(f"    Flagged subjects (n={len(flagged_vals)}): mean={np.mean(flagged_vals):.4f}, "
          f"values={[round(v,4) for v in flagged_vals]}")
    print(f"    Non-flagged subjects (n={len(non_flagged_vals)}): mean={np.mean(non_flagged_vals):.4f}")

    if len(flagged_vals) >= 2 and len(non_flagged_vals) >= 2:
        t_stat, p_val = ttest_ind(flagged_vals, non_flagged_vals, equal_var=False)
        print(f"    Welch's t-test: t={t_stat:.3f}, p={p_val:.4g}")
        if p_val < 0.05 and np.mean(flagged_vals) > np.mean(non_flagged_vals):
            print(f"    -> SUPPORTS the ocular-artifact hypothesis: {ch} delta power is significantly "
                  f"elevated in the flagged subjects.")
        elif p_val < 0.05:
            print(f"    -> Significant, but LOWER delta power in flagged subjects -- does NOT support "
                  f"the ocular hypothesis; investigate another cause.")
        else:
            print(f"    -> Not statistically significant at group level (n={len(flagged_vals)} is small, "
                  f"so this test has limited power) -- rely more on Test A (within-subject) above.")
    else:
        print(f"    -> Too few flagged subjects (n={len(flagged_vals)}) for a reliable group t-test; "
              f"interpret Test A (within-subject) results instead.")

print(f"\n{'='*90}\nSUMMARY / INTERPRETATION GUIDE\n{'='*90}")
print("""
- If Test A shows 'ELEVATED' (ratio > 1.5x) for most/all flagged subjects, AND Test B is
  significant (or trends strongly the same direction despite small n), this is solid evidence
  the low-correlation flag at Fp1/Fp2 reflects genuine residual ocular contamination, not a
  coincidence or generic electrode issue.
- If ratios are 'comparable' or 'REDUCED', the ocular hypothesis is not supported either, and the
  cause is more likely electrode impedance/placement specific to those subjects -- worth a manual
  visual check of time_series_snippet.png for exactly these subject/channel pairs.
""")

Checked 25 subjects.


Fp1: flagged as low-correlation in 1 subject(s): ['VP019']

  TEST A (within-subject): channel delta power vs. that subject's other channels
    VP019: Fp1 delta rel.power=15.4907 vs subject's other-channel mean=12.1959 (ratio=1.27x) -> comparable

  TEST B (between-subject): Fp1 delta power, flagged vs non-flagged subjects
    Flagged subjects (n=1): mean=15.4907, values=[np.float64(15.4907)]
    Non-flagged subjects (n=24): mean=11.4923
    -> Too few flagged subjects (n=1) for a reliable group t-test; interpret Test A (within-subject) results instead.

Fp2: flagged as low-correlation in 0 subject(s): []
  No subjects flagged this channel -- nothing to test.

SUMMARY / INTERPRETATION GUIDE

- If Test A shows 'ELEVATED' (ratio > 1.5x) for most/all flagged subjects, AND Test B is
  significant (or trends strongly the same direction despite small n), this is solid evidence
  the low-correlation flag at Fp1/Fp2 reflects genuine residual ocular contamination, not a


In [ ]:
#one fnirs check

In [9]:
# این رو توی یه سلول ساده اجرا کنید
import numpy as np, os

OUTPUT_PATH = r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"
issues = []
for n in range(1, 27):
    if n == 4:
        continue
    sid = f"VP{n:03d}"
    path = os.path.join(OUTPUT_PATH, sid, f"{sid}_fnirs_windows.npy")
    if not os.path.exists(path):
        issues.append(f"{sid}: file missing")
        continue
    d = np.load(path)
    if np.isnan(d).any():
        issues.append(f"{sid}: NaN detected")
    if d.std() < 0.01:
        issues.append(f"{sid}: suspiciously low variance (std={d.std():.4f})")

print("Issues found:" if issues else "All fNIRS files: clean (no NaN, variance OK)")
for i in issues: print(" ", i)

All fNIRS files: clean (no NaN, variance OK)
